In [17]:

from google.colab import files

uploaded = files.upload()


Saving 1794638551800_emberwise_stage_a_v3 (1).zip to 1794638551800_emberwise_stage_a_v3 (1) (1).zip


In [18]:

!unzip -q -o 1794638551800_emberwise_stage_a_v3.zip -d /content/

import os

print(os.listdir("/content"))


unzip:  cannot find or open 1794638551800_emberwise_stage_a_v3.zip, 1794638551800_emberwise_stage_a_v3.zip.zip or 1794638551800_emberwise_stage_a_v3.zip.ZIP.
['.config', '1794638551800_emberwise_stage_a_v3 (1).zip', '1794638551800_emberwise_stage_a_v3 (1) (1).zip', 'sample_data']


In [19]:

%cd /content/emberwise

import os
print("Project folder:", os.getcwd())
print("Pipeline exists:", os.path.isdir("pipeline"))


[Errno 2] No such file or directory: '/content/emberwise'
/content
Project folder: /content
Pipeline exists: False


In [20]:

!pip install -q polars numpy scipy h3 jsonschema pytest pyarrow


In [21]:

!python -m pipeline.tables.make_mocks

!ls -lh data/processed/


/usr/bin/python3: Error while finding module specification for 'pipeline.tables.make_mocks' (ModuleNotFoundError: No module named 'pipeline')
ls: cannot access 'data/processed/': No such file or directory


In [22]:

import os
import sys

sys.path.insert(0, "/content/emberwise")

for folder in ["pipeline/calibrate", "pipeline/validate"]:
    os.makedirs(folder, exist_ok=True)
    init_file = os.path.join(folder, "__init__.py")
    open(init_file, "a").close()

print("M2 folders created successfully.")


M2 folders created successfully.


In [23]:

%%writefile pipeline/calibrate/ratios.py

"""M2: MODIS to VIIRS conversion ratio calibration."""

import numpy as np

K = 20

YEARS = list(range(2012, 2020))

BASE_REGIONS = ["punjab", "haryana"]

PRESETS = {
    "punjab": [0],
    "haryana": [1],
    "punjab_haryana": [0, 1]
}


def build_cubes(
    path,
    sensors=("VIIRS_SNPP", "MODIS_ALL", "TERRA"),
    years=YEARS
):
    """Build [region, zone, year, month] arrays from tile_day.parquet."""

    import polars as pl

    td = pl.read_parquet(path)

    zones = sorted(
        td.filter(
            pl.col("region").is_in(BASE_REGIONS)
        )["zone_r3"].unique().to_list()
    )

    zone_index = {z: i for i, z in enumerate(zones)}

    cubes = {
        sensor: np.zeros(
            (len(BASE_REGIONS), len(zones), len(years), 12),
            dtype=float
        )
        for sensor in sensors
    }

    grouped = (
        td.filter(
            pl.col("region").is_in(BASE_REGIONS)
            & (pl.col("product") == "SP")
            & pl.col("sensor_set").is_in(list(sensors))
            & pl.col("date").dt.year().is_between(
                years[0], years[-1]
            )
        )
        .group_by(
            "region",
            "zone_r3",
            "sensor_set",
            pl.col("date").dt.year().alias("year"),
            pl.col("date").dt.month().alias("month")
        )
        .agg(
            pl.col("n_tile_day").sum().alias("total")
        )
    )

    for region, zone, sensor, year, month, total in grouped.iter_rows():

        # Exclude partial January 2012.
        if (year, month) == (2012, 1):
            continue

        ri = BASE_REGIONS.index(region)
        zi = zone_index[zone]
        yi = year - years[0]

        cubes[sensor][ri, zi, yi, month - 1] = total

    return cubes, zones


def fit_ratio(V, M, k=K):
    """Fit shrinkage ratios.

    V and M shapes: [zone, year, month].
    """

    V = np.asarray(V, dtype=float)
    M = np.asarray(M, dtype=float)

    if V.shape != M.shape or V.ndim != 3:
        raise ValueError("V and M must have matching [zone, year, month] shapes.")

    if k < 0:
        raise ValueError("k must be non-negative.")

    v_region = V.sum(axis=(0, 1))
    m_region = M.sum(axis=(0, 1))

    r_region = np.divide(
        v_region,
        m_region,
        out=np.full_like(v_region, np.nan),
        where=m_region > 0
    )

    # Use zero contribution for months without regional observations.
    prior = np.nan_to_num(r_region, nan=0.0)

    r_hat = np.divide(
        V.sum(axis=1) + k * prior,
        M.sum(axis=1) + k,
        out=np.broadcast_to(prior, (V.shape[0], V.shape[2])).copy(),
        where=(M.sum(axis=1) + k) > 0
    )

    yearly = np.divide(
        V + k * prior[None, None, :],
        M + k,
        out=np.broadcast_to(
            prior[None, None, :], V.shape
        ).copy(),
        where=(M + k) > 0
    )

    return r_hat, yearly, r_region


def ratio_range(r_hat, yearly):
    """Calculate 10th to 90th percentile range."""

    lo = np.nanpercentile(yearly, 10, axis=1)
    hi = np.nanpercentile(yearly, 90, axis=1)

    lo = np.minimum(lo, r_hat)
    hi = np.maximum(hi, r_hat)

    return lo, hi


Writing pipeline/calibrate/ratios.py


In [24]:

%%writefile pipeline/validate/hide_a_year.py

"""M2: Hide-a-year validation."""

import numpy as np

from pipeline.calibrate.ratios import (
    fit_ratio,
    ratio_range,
    PRESETS,
    YEARS
)

OCT_NOV = [9, 10]


def hide_a_year(V, M, years=YEARS, k=20):
    """Learn on all but one year and predict the hidden year."""

    V = np.asarray(V, dtype=float)
    M = np.asarray(M, dtype=float)

    if V.shape != M.shape or V.ndim != 4:
        raise ValueError(
            "V and M must have matching [region, zone, year, month] shapes."
        )

    if V.shape[2] != len(years):
        raise ValueError("Number of years does not match the data.")

    rows = []

    Vz = V.sum(axis=0)
    Mz = M.sum(axis=0)

    for i, year in enumerate(years):

        keep = [j for j in range(len(years)) if j != i]

        r_hat, yearly, r_region = fit_ratio(
            Vz[:, keep],
            Mz[:, keep],
            k
        )

        lo, hi = ratio_range(r_hat, yearly)

        total_v = Vz[:, keep].sum()
        total_m = Mz[:, keep].sum()

        global_ratio = (
            total_v / total_m if total_m > 0 else np.nan
        )

        for pi, (preset, parts) in enumerate(PRESETS.items()):

            m = M[parts].sum(axis=0)[:, i]

            actual = (
                V[parts].sum(axis=0)[:, i].sum(axis=0)
            )

            pred = np.nansum(m * r_hat, axis=0)
            pred_lo = np.nansum(m * lo, axis=0)
            pred_hi = np.nansum(m * hi, axis=0)

            rows.append({
                "region": pi,
                "preset": preset,
                "year": year,
                "actual": actual,
                "pred": pred,
                "pred_lo": pred_lo,
                "pred_hi": pred_hi,
                "base_none": m.sum(axis=0),
                "base_global": m.sum(axis=0) * global_ratio
            })

    return rows


def pct_err(pred, actual):

    if actual == 0:
        return np.nan

    return 100.0 * (pred - actual) / actual


def summarize(
    rows,
    months=None,
    preset="punjab_haryana"
):

    if months is None:
        months = OCT_NOV

    output = []

    for row in rows:

        if row["preset"] != preset:
            continue

        actual = row["actual"][months].sum()
        pred = row["pred"][months].sum()

        lower = row["pred_lo"][months].sum()
        upper = row["pred_hi"][months].sum()

        output.append({
            "year": row["year"],
            "actual": actual,
            "pred": pred,
            "err": pct_err(pred, actual),
            "in_range": bool(lower <= actual <= upper),
            "err_none": pct_err(
                row["base_none"][months].sum(),
                actual
            ),
            "err_global": pct_err(
                row["base_global"][months].sum(),
                actual
            )
        })

    return output


def error_by_level(table):

    ordered = sorted(
        table,
        key=lambda row: row["actual"]
    )

    n = len(ordered)

    cut1 = round(n / 3)
    cut2 = round(2 * n / 3)

    groups = {
        "low": ordered[:cut1],
        "mid": ordered[cut1:cut2],
        "high": ordered[cut2:]
    }

    result = {}

    for name, group in groups.items():

        if not group:
            continue

        errors = np.array(
            [row["err"] for row in group],
            dtype=float
        )

        result[name] = (
            float(np.nanmean(np.abs(errors))),
            float(np.nanmean(errors))
        )

    return result


Writing pipeline/validate/hide_a_year.py


In [25]:

import importlib

import pipeline.calibrate.ratios as ratios
import pipeline.validate.hide_a_year as validation

importlib.reload(ratios)
importlib.reload(validation)

print("fit_ratio:", callable(ratios.fit_ratio))
print("build_cubes:", callable(ratios.build_cubes))
print("hide_a_year:", callable(validation.hide_a_year))
print("summarize:", callable(validation.summarize))
print("error_by_level:", callable(validation.error_by_level))


fit_ratio: True
build_cubes: True
hide_a_year: True
summarize: True
error_by_level: True


In [26]:

import os

print("Current directory:", os.getcwd())
print("Files and folders:")

for item in os.listdir("/content"):
    print(item)


Current directory: /content
Files and folders:
.config
pipeline
1794638551800_emberwise_stage_a_v3 (1).zip
1794638551800_emberwise_stage_a_v3 (1) (1).zip
sample_data


In [27]:

import os

matches = []

for root, dirs, files in os.walk("/content"):
    if "pipeline" in dirs:
        matches.append(os.path.join(root, "pipeline"))

print("Pipeline folders found:")

for path in matches:
    print(path)


Pipeline folders found:
/content/pipeline


In [30]:

import os
import sys

matches = []

for root, dirs, files in os.walk("/content"):
    if "pipeline" in dirs:
        matches.append(os.path.join(root, "pipeline"))

if not matches:
    raise FileNotFoundError(
        "pipeline folder not found. Upload and extract the Stage A ZIP first."
    )

PROJECT_PATH = os.path.dirname(matches[0])

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

os.chdir(PROJECT_PATH)

print("Project path:", PROJECT_PATH)
print("Pipeline folder exists:", os.path.isdir("pipeline"))
print("Calibration folder:", os.path.isdir("pipeline/calibrate"))
print("Validation folder:", os.path.isdir("pipeline/validate"))


Project path: /content
Pipeline folder exists: True
Calibration folder: True
Validation folder: True


In [31]:

import os

for path in [
    "pipeline/calibrate/ratios.py",
    "pipeline/validate/hide_a_year.py"
]:
    print(path, "EXISTS:", os.path.isfile(path))


pipeline/calibrate/ratios.py EXISTS: True
pipeline/validate/hide_a_year.py EXISTS: True


In [32]:

import numpy as np

from pipeline.calibrate.ratios import fit_ratio
from pipeline.validate.hide_a_year import (
    hide_a_year,
    summarize,
    error_by_level
)

print("SUCCESS: All M2 modules imported!")


SUCCESS: All M2 modules imported!


In [34]:
import os

print("Project folder:", os.getcwd())
print("Mock data folder exists:", os.path.exists("data/processed"))
print("tile_day.parquet exists:",
      os.path.exists("data/processed/tile_day.parquet"))
print("M3 conversion file exists:",
      os.path.exists("web/public/data/conversion.json"))

Project folder: /content
Mock data folder exists: False
tile_day.parquet exists: False
M3 conversion file exists: False


In [35]:
!python -m pipeline.tables.make_mocks

/usr/bin/python3: Error while finding module specification for 'pipeline.tables.make_mocks' (ModuleNotFoundError: No module named 'pipeline.tables')


In [36]:
import os

print("Folders in /content:")
print(os.listdir("/content"))

print("\nSearching for Emberwise project files:")
for root, dirs, files in os.walk("/content"):
    if "requirements-tables.txt" in files:
        print("Found project at:", root)

Folders in /content:
['.config', 'pipeline', '1794638551800_emberwise_stage_a_v3 (1).zip', '1794638551800_emberwise_stage_a_v3 (1) (1).zip', 'sample_data']

Searching for Emberwise project files:


In [37]:
import os
import zipfile

print("=== CONTENTS OF /content/pipeline ===")
for root, dirs, files in os.walk("/content/pipeline"):
    level = root.replace("/content/pipeline", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files[:20]:
        print(f"{indent}  {file}")

print("\n=== ZIP FILE CONTENTS ===")
zip_path = "/content/1794638551800_emberwise_stage_a_v3 (1).zip"

with zipfile.ZipFile(zip_path, "r") as z:
    for name in z.namelist()[:60]:
        print(name)

=== CONTENTS OF /content/pipeline ===
pipeline/
  validate/
    __init__.py
    hide_a_year.py
    __pycache__/
      hide_a_year.cpython-313.pyc
      __init__.cpython-313.pyc
  calibrate/
    __init__.py
    ratios.py
    __pycache__/
      ratios.cpython-313.pyc
      __init__.cpython-313.pyc

=== ZIP FILE CONTENTS ===
emberwise/
emberwise/Makefile
emberwise/README_TABLES.md
emberwise/web/
emberwise/web/public/
emberwise/web/public/data/
emberwise/web/public/data/tile_month_2011.json
emberwise/web/public/data/tile_month_2005.json
emberwise/web/public/data/hourly.json
emberwise/web/public/data/tile_month_2007.json
emberwise/web/public/data/tile_month_2004.json
emberwise/web/public/data/tile_month_2003.json
emberwise/web/public/data/tile_month_2015.json
emberwise/web/public/data/tile_month_2023.json
emberwise/web/public/data/tile_month_2009.json
emberwise/web/public/data/relay_sim.json
emberwise/web/public/data/climatology.json
emberwise/web/public/data/tile_month_2002.json
emberwise/

In [38]:
import os
import zipfile

ZIP_PATH = "/content/1794638551800_emberwise_stage_a_v3 (1).zip"
PROJECT_PATH = "/content/emberwise"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall("/content")

print("Project extracted:", os.path.isdir(PROJECT_PATH))
print("Tables folder exists:",
      os.path.isdir(PROJECT_PATH + "/pipeline/tables"))
print("Mock builder exists:",
      os.path.isfile(PROJECT_PATH + "/pipeline/tables/make_mocks.py"))
print("M2 calibrate folder still exists:",
      os.path.isdir("/content/pipeline/calibrate"))

Project extracted: True
Tables folder exists: True
Mock builder exists: True
M2 calibrate folder still exists: True


In [39]:
import os
import sys

PROJECT_PATH = "/content/emberwise"
os.chdir(PROJECT_PATH)

print("Current folder:", os.getcwd())
print("Tables module exists:",
      os.path.isfile("pipeline/tables/make_mocks.py"))
print("README exists:",
      os.path.isfile("README_TABLES.md"))

Current folder: /content/emberwise
Tables module exists: True
README exists: True


In [40]:
import os

os.chdir("/content/emberwise")

print("=== PROJECT ROOT FILES ===")
print(os.listdir("."))

print("\n=== README_TABLES.md ===")
with open("README_TABLES.md", "r") as f:
    print(f.read())

=== PROJECT ROOT FILES ===
['pipeline', 'Makefile.tables', '.gitignore', 'tests', 'requirements-tables.txt', 'web', 'README_TABLES.md', 'CHANGELOG.md', 'Makefile', 'conftest.py', 'docs']

=== README_TABLES.md ===
# EMBERWISE: analytics tables, schema and checks (M3)

    pip install -r requirements-tables.txt
    make tables        # build tables from the hand-offs -> docs/schema.md -> contract check -> size report
    make test-tables   # the above, then pytest
    make mocks         # rewrite the mock hand-offs (only needed while real files are missing)

`make tables` creates mock hand-offs automatically if `data/processed/tile_day.parquet` is missing.
When M1 and M2 drop real files in place, the same command builds the real tables.

| path | what |
|---|---|
| `pipeline/tables/config.py` | every tunable setting |
| `pipeline/tables/contracts.py` | the schema, single source of truth |
| `pipeline/tables/mocks.py` | mock stand-ins for M1, M2, M5 and M6 hand-offs |
| `pipeline/tables/b

In [41]:
import os
import subprocess
import sys

os.chdir("/content/emberwise")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements-tables.txt"
])

print("SUCCESS: Project dependencies installed.")

SUCCESS: Project dependencies installed.


In [42]:
import os

os.chdir("/content/emberwise")

files_to_inspect = [
    "pipeline/tables/make_mocks.py",
    "pipeline/tables/mocks.py",
    "Makefile",
    "Makefile.tables"
]

for path in files_to_inspect:
    print("\n" + "=" * 60)
    print("FILE:", path)
    print("=" * 60)

    if os.path.exists(path):
        with open(path, "r") as f:
            content = f.read()
        print(content[:12000])
    else:
        print("File not found.")


FILE: pipeline/tables/make_mocks.py
"""Write mock stand-ins for every upstream hand-off.  Usage: python -m pipeline.tables.make_mocks"""
from . import mocks

if __name__ == "__main__":
    mocks.generate()
    print("wrote mock hand-offs: data/processed/*.parquet and upstream JSON in web/public/data")


FILE: pipeline/tables/mocks.py
"""Mock stand-ins for everything M3 receives from M1, M2, M5 and M6.

Writes data/processed/{tiles,tile_day}.parquet and the upstream JSON contracts
(regions, coverage, hourly, drift, conversion, validation, relay_sim, hotspot CSV
headers). M3's own tables are then built from these by build.py, exactly as they
will be from the real hand-offs.

Everything is "mock": true. Behaviour is meant to be realistic (Oct-Nov peak,
smaller Apr-May peak, quiet months, saturated days, VIIRS absent before 2012,
a jump in raw counts at 2012, NRT newest months, a data gap, a low-fire control
district). The VALUES are invented and must never be quoted.
"""
from __future__ 

m2 work


In [43]:
import os
import shutil

SOURCE = "/content/emberwise"
WORK = "/content/emberwise_m2_work"

if not os.path.exists(WORK):
    shutil.copytree(SOURCE, WORK)

print("Working copy exists:", os.path.isdir(WORK))
print("Mock generator exists:",
      os.path.isfile(WORK + "/pipeline/tables/make_mocks.py"))
print("Original M3 project preserved:",
      os.path.isdir(SOURCE + "/pipeline/tables"))

Working copy exists: True
Mock generator exists: True
Original M3 project preserved: True


In [44]:
import os
import sys
import subprocess

os.chdir("/content/emberwise_m2_work")

result = subprocess.run(
    [sys.executable, "-m", "pipeline.tables.make_mocks"],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("Output:")
print(result.stdout)

if result.stderr:
    print("Errors:")
    print(result.stderr)

Return code: 0
Output:
wrote mock hand-offs: data/processed/*.parquet and upstream JSON in web/public/data



In [45]:
import os

os.chdir("/content/emberwise_m2_work")

files = [
    "data/processed/tiles.parquet",
    "data/processed/tile_day.parquet",
    "web/public/data/coverage.json",
    "web/public/data/conversion.json",
    "web/public/data/validation.json",
    "web/public/data/relay_sim.json"
]

for path in files:
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f"{'OK' if exists else 'MISSING'} | {path} | {size:,} bytes")

OK | data/processed/tiles.parquet | 6,080 bytes
OK | data/processed/tile_day.parquet | 7,644,293 bytes
OK | web/public/data/coverage.json | 492 bytes
OK | web/public/data/conversion.json | 26,880 bytes
OK | web/public/data/validation.json | 2,381 bytes
OK | web/public/data/relay_sim.json | 1,448 bytes


In [46]:
import os
import polars as pl
import json

os.chdir("/content/emberwise_m2_work")

# Load the two datasets
tiles = pl.read_parquet("data/processed/tiles.parquet")
tile_day = pl.read_parquet("data/processed/tile_day.parquet")

print("========== TILES DATASET ==========")
print("Shape:", tiles.shape)
print("Columns:", tiles.columns)
print(tiles.head(5))

print("\n========== TILE_DAY DATASET ==========")
print("Shape:", tile_day.shape)
print("Columns:", tile_day.columns)
print(tile_day.head(5))

print("\n========== DATA TYPES ==========")
print(tile_day.schema)

print("\n========== SENSOR SETS ==========")
print(tile_day.group_by("sensor_set").agg(
    pl.len().alias("rows")
).sort("sensor_set"))

print("\n========== REGIONS ==========")
print(tile_day.group_by("region").agg(
    pl.len().alias("rows")
).sort("region"))

print("\n========== DATE RANGE ==========")
print("First date:", tile_day["date"].min())
print("Last date:", tile_day["date"].max())

print("\n========== COVERAGE.JSON ==========")
with open("web/public/data/coverage.json") as f:
    coverage = json.load(f)
print(json.dumps(coverage, indent=2)[:4000])

========== TILES DATASET ==========
Shape: (3233, 3)
Columns: ['region', 'h3_r6', 'zone_r3']
shape: (5, 3)
┌────────┬─────────────────┬─────────────────┐
│ region ┆ h3_r6           ┆ zone_r3         │
│ ---    ┆ ---             ┆ ---             │
│ str    ┆ str             ┆ str             │
╞════════╪═════════════════╪═════════════════╡
│ punjab ┆ 863d10d07ffffff ┆ 833d10fffffffff │
│ punjab ┆ 863d10d17ffffff ┆ 833d10fffffffff │
│ punjab ┆ 863d10d27ffffff ┆ 833d10fffffffff │
│ punjab ┆ 863d10d37ffffff ┆ 833d10fffffffff │
│ punjab ┆ 863d14007ffffff ┆ 833d14fffffffff │
└────────┴─────────────────┴─────────────────┘

========== TILE_DAY DATASET ==========
Shape: (1930899, 9)
Columns: ['region', 'h3_r6', 'zone_r3', 'date', 'sensor_set', 'product', 'frp_sum', 'n_hotspots', 'n_tile_day']
shape: (5, 9)
┌────────┬─────────────┬────────────┬────────────┬───┬─────────┬─────────┬────────────┬────────────┐
│ region ┆ h3_r6       ┆ zone_r3    ┆ date       ┆ … ┆ product ┆ frp_sum ┆ n_hotspots ┆ n

In [47]:
import os
import json

WORK = "/content/emberwise_m2_work"

files_to_read = [
    "pipeline/tables/config.py",
    "pipeline/calibrate/ratios.py",
    "pipeline/validate/hide_a_year.py"
]

for path in files_to_read:
    full_path = os.path.join(WORK, path)

    print("\n" + "=" * 65)
    print("FILE:", path)
    print("=" * 65)

    if os.path.exists(full_path):
        with open(full_path, "r") as f:
            print(f.read())
    else:
        print("NOT FOUND in working copy")

print("\n" + "=" * 65)
print("COVERAGE DETAILS")
print("=" * 65)

with open(os.path.join(WORK, "web/public/data/coverage.json")) as f:
    coverage = json.load(f)

print("Epoch:", coverage.get("epoch"))
print("Sensor sets:", coverage.get("sensor_sets"))
print("Products:", coverage.get("products"))
print("Coverage columns:")
print(json.dumps(coverage.get("columns"), indent=2))


FILE: pipeline/tables/config.py
"""EMBERWISE M3: every tunable setting lives here. Nothing else hard-codes a threshold.

Section numbers refer to "EMBERWISE Final Build Plan v4". Items marked [verify]
are unconfirmed in the plan itself or are M3 assumptions the plan does not state.
"""
from pathlib import Path

# --- Schema (plan s5: frozen at H2 as 1.0) ---------------------------------
SCHEMA_VERSION = "1.0"

# --- Paths (plan s5 repo layout) -------------------------------------------
ROOT = Path(__file__).resolve().parents[2]
DATA_DIR = ROOT / "web" / "public" / "data"        # shipped JSON
CSV_DIR = DATA_DIR / "hotspots"                     # lazy hotspot CSVs, budgeted separately
PROCESSED_DIR = ROOT / "data" / "processed"         # M1 hand-off (git-ignored)
DOCS_DIR = ROOT / "docs"

# --- Grid (plan s2) ---------------------------------------------------------
H3_RES = 6
H3_FALLBACK_RES = 5
ZONE_RES = 3
SENSITIVITY_RES = [5, 6, 7]

# --- Time (plan s2 eras) --------------------

In [48]:
import os
import shutil

SOURCE = "/content/pipeline"
WORK_PIPELINE = "/content/emberwise_m2_work/pipeline"

for folder in ["calibrate", "validate"]:
    source_folder = os.path.join(SOURCE, folder)
    destination_folder = os.path.join(WORK_PIPELINE, folder)

    if not os.path.isdir(source_folder):
        raise FileNotFoundError(f"Missing M2 folder: {source_folder}")

    shutil.copytree(
        source_folder,
        destination_folder,
        dirs_exist_ok=True
    )

    print(f"Copied: {folder}")

print("\nM2 module check:")
for path in [
    "calibrate/ratios.py",
    "validate/hide_a_year.py"
]:
    full_path = os.path.join(WORK_PIPELINE, path)
    print(path, "EXISTS" if os.path.isfile(full_path) else "MISSING")

Copied: calibrate
Copied: validate

M2 module check:
calibrate/ratios.py EXISTS
validate/hide_a_year.py EXISTS


In [49]:
import os

os.chdir("/content/emberwise_m2_work")

for path in [
    "pipeline/calibrate/ratios.py",
    "pipeline/validate/hide_a_year.py"
]:
    print("\n" + "=" * 60)
    print("FILE:", path)
    print("=" * 60)

    with open(path, "r") as f:
        print(f.read())


FILE: pipeline/calibrate/ratios.py

"""M2: MODIS to VIIRS conversion ratio calibration."""

import numpy as np

K = 20

YEARS = list(range(2012, 2020))

BASE_REGIONS = ["punjab", "haryana"]

PRESETS = {
    "punjab": [0],
    "haryana": [1],
    "punjab_haryana": [0, 1]
}


def build_cubes(
    path,
    sensors=("VIIRS_SNPP", "MODIS_ALL", "TERRA"),
    years=YEARS
):
    """Build [region, zone, year, month] arrays from tile_day.parquet."""

    import polars as pl

    td = pl.read_parquet(path)

    zones = sorted(
        td.filter(
            pl.col("region").is_in(BASE_REGIONS)
        )["zone_r3"].unique().to_list()
    )

    zone_index = {z: i for i, z in enumerate(zones)}

    cubes = {
        sensor: np.zeros(
            (len(BASE_REGIONS), len(zones), len(years), 12),
            dtype=float
        )
        for sensor in sensors
    }

    grouped = (
        td.filter(
            pl.col("region").is_in(BASE_REGIONS)
            & (pl.col("product") == "SP")
         

In [50]:
import os
import json
from datetime import date, timedelta

os.chdir("/content/emberwise_m2_work")

with open("web/public/data/coverage.json") as f:
    coverage = json.load(f)

epoch = date.fromisoformat(coverage["epoch"])

sensor_names = coverage["sensor_sets"]
product_names = coverage["products"]
c = coverage["columns"]

def get_intervals(sensor, product):
    si = sensor_names.index(sensor)
    pi = product_names.index(product)

    intervals = []

    for i in range(len(c["sensor_set"])):
        if (
            c["sensor_set"][i] == si
            and c["product"][i] == pi
            and c["has_data"][i]
        ):
            start = epoch + timedelta(days=c["start"][i])
            end = epoch + timedelta(days=c["end"][i])
            intervals.append((start, end))

    return intervals

modis = get_intervals("MODIS_ALL", "SP")
viirs = get_intervals("VIIRS_SNPP", "SP")

print("MODIS_ALL SP coverage:")
for start, end in modis:
    print(start, "to", end)

print("\nVIIRS_SNPP SP coverage:")
for start, end in viirs:
    print(start, "to", end)

print("\nOVERLAPPING COVERAGE:")
for ms, me in modis:
    for vs, ve in viirs:
        start = max(ms, vs)
        end = min(me, ve)

        if start <= end:
            print(start, "to", end)

print("\nLEARNING WINDOW:", "2012-01-01 to 2019-12-31")

MODIS_ALL SP coverage:
2000-11-01 to 2005-07-09
2005-07-13 to 2026-08-31

VIIRS_SNPP SP coverage:
2012-01-20 to 2026-08-31

OVERLAPPING COVERAGE:
2012-01-20 to 2026-08-31

LEARNING WINDOW: 2012-01-01 to 2019-12-31


In [51]:
import os
import sys

os.chdir("/content/emberwise_m2_work")

if "/content/emberwise_m2_work" not in sys.path:
    sys.path.insert(0, "/content/emberwise_m2_work")

from pipeline.calibrate.ratios import (
    build_cubes, fit_ratio, ratio_range
)

from pipeline.validate.hide_a_year import (
    hide_a_year, summarize, error_by_level
)

print("SUCCESS: M2 modules imported.")

SUCCESS: M2 modules imported.


In [52]:
from pathlib import Path

path = Path("pipeline/calibrate/ratios.py")
source = path.read_text()

start = source.index("def build_cubes(")
end = source.index("\ndef fit_ratio(", start)

new_function = '''
def build_cubes(
    path,
    coverage_path="web/public/data/coverage.json",
    sensors=("VIIRS_SNPP", "MODIS_ALL", "TERRA"),
    years=YEARS
):
    """Build coverage-aware [region, zone, year, month] arrays."""

    import json
    import polars as pl
    from datetime import date, timedelta

    td = pl.read_parquet(path)

    with open(coverage_path) as f:
        coverage = json.load(f)

    epoch = date.fromisoformat(coverage["epoch"])
    sensor_names = coverage["sensor_sets"]
    product_names = coverage["products"]
    c = coverage["columns"]

    valid_intervals = {}

    for sensor in sensors:
        si = sensor_names.index(sensor)
        pi = product_names.index("SP")

        intervals = []

        for i in range(len(c["sensor_set"])):
            if (
                c["sensor_set"][i] == si
                and c["product"][i] == pi
                and c["has_data"][i]
            ):
                intervals.append((
                    epoch + timedelta(days=c["start"][i]),
                    epoch + timedelta(days=c["end"][i])
                ))

        valid_intervals[sensor] = intervals

    zones = sorted(
        td.filter(
            pl.col("region").is_in(BASE_REGIONS)
        )["zone_r3"].unique().to_list()
    )

    zone_index = {z: i for i, z in enumerate(zones)}

    cubes = {
        sensor: np.zeros(
            (len(BASE_REGIONS), len(zones), len(years), 12),
            dtype=float
        )
        for sensor in sensors
    }

    grouped = (
        td.filter(
            pl.col("region").is_in(BASE_REGIONS)
            & (pl.col("product") == "SP")
            & pl.col("sensor_set").is_in(list(sensors))
            & pl.col("date").dt.year().is_between(
                years[0], years[-1]
            )
        )
        .group_by(
            "region",
            "zone_r3",
            "sensor_set",
            pl.col("date").dt.year().alias("year"),
            pl.col("date").dt.month().alias("month")
        )
        .agg(
            pl.col("n_tile_day").sum().alias("total")
        )
    )

    for region, zone, sensor, year, month, total in grouped.iter_rows():

        if (year, month) == (2012, 1):
            continue

        month_start = date(year, month, 1)

        if month == 12:
            month_end = date(year + 1, 1, 1) - timedelta(days=1)
        else:
            month_end = date(year, month + 1, 1) - timedelta(days=1)

        # Keep only months fully covered by this sensor.
        covered = any(
            start <= month_start and end >= month_end
            for start, end in valid_intervals[sensor]
        )

        if not covered:
            continue

        ri = BASE_REGIONS.index(region)
        zi = zone_index[zone]
        yi = year - years[0]

        cubes[sensor][ri, zi, yi, month - 1] = total

    return cubes, zones
'''

path.write_text(source[:start] + new_function + source[end:])

print("Updated build_cubes() with coverage checks.")

Updated build_cubes() with coverage checks.


In [53]:
import os
import sys
import numpy as np

os.chdir("/content/emberwise_m2_work")

if "/content/emberwise_m2_work" not in sys.path:
    sys.path.insert(0, "/content/emberwise_m2_work")

from pipeline.calibrate.ratios import build_cubes

cubes, zones = build_cubes(
    "data/processed/tile_day.parquet"
)

print("Zones:", len(zones))
print("Zone names:", zones[:10])

for sensor, cube in cubes.items():
    print(
        sensor,
        "shape:", cube.shape,
        "total:", round(float(cube.sum()), 2),
        "nonzero months:", int(np.count_nonzero(cube.sum(axis=(0, 1, 2))))
    )

print("\nMODIS and VIIRS cube shapes match:",
      cubes["MODIS_ALL"].shape == cubes["VIIRS_SNPP"].shape)

Zones: 16
Zone names: ['833d10fffffffff', '833d12fffffffff', '833d14fffffffff', '833d16fffffffff', '833d32fffffffff', '833d33fffffffff', '833da0fffffffff', '833da1fffffffff', '833da3fffffffff', '833da4fffffffff']
VIIRS_SNPP shape: (2, 16, 8, 12) total: 341629.0 nonzero months: 12
MODIS_ALL shape: (2, 16, 8, 12) total: 196070.0 nonzero months: 12
TERRA shape: (2, 16, 8, 12) total: 108083.0 nonzero months: 12

MODIS and VIIRS cube shapes match: True


In [54]:
import numpy as np

from pipeline.calibrate.ratios import (
    fit_ratio,
    ratio_range,
    K
)

# Keep MODIS and VIIRS separate.
V = cubes["VIIRS_SNPP"].sum(axis=0)
M = cubes["MODIS_ALL"].sum(axis=0)

print("Input shapes:")
print("VIIRS:", V.shape)
print("MODIS:", M.shape)

# Fit conversion ratios.
r_hat, yearly, r_region = fit_ratio(V, M, k=K)

# Calculate the uncertainty range.
lo, hi = ratio_range(r_hat, yearly)

print("\nOutput shapes:")
print("Monthly fitted ratios:", r_hat.shape)
print("Yearly ratios:", yearly.shape)
print("Regional ratios:", r_region.shape)
print("Lower range:", lo.shape)
print("Upper range:", hi.shape)

print("\nRatio diagnostics:")
print("Finite fitted ratios:", np.isfinite(r_hat).sum())
print("Missing fitted ratios:", np.isnan(r_hat).sum())
print("Regional monthly ratios:", np.round(r_region, 3))

valid = r_hat[np.isfinite(r_hat)]

if len(valid):
    print("\nFitted ratio percentiles:")
    print("10th:", round(float(np.percentile(valid, 10)), 3))
    print("50th:", round(float(np.percentile(valid, 50)), 3))
    print("90th:", round(float(np.percentile(valid, 90)), 3))
else:
    print("WARNING: No finite fitted ratios.")

Input shapes:
VIIRS: (16, 8, 12)
MODIS: (16, 8, 12)

Output shapes:
Monthly fitted ratios: (16, 12)
Yearly ratios: (16, 8, 12)
Regional ratios: (12,)
Lower range: (16, 12)
Upper range: (16, 12)

Ratio diagnostics:
Finite fitted ratios: 192
Missing fitted ratios: 0
Regional monthly ratios: [1.677 1.792 1.857 1.73  1.747 1.815 1.716 1.833 1.737 1.741 1.743 1.695]

Fitted ratio percentiles:
10th: 1.645
50th: 1.747
90th: 1.888


In [55]:
from pipeline.validate.hide_a_year import (
    hide_a_year,
    summarize,
    error_by_level
)

# Hold out each year in turn.
validation_rows = hide_a_year(
    cubes["VIIRS_SNPP"],
    cubes["MODIS_ALL"],
    k=K
)

print("Validation rows:", len(validation_rows))
print("First result:")
print(validation_rows[0])

# Summarize October and November for both regions.
summary = summarize(
    validation_rows,
    months=[9, 10],
    preset="punjab_haryana"
)

print("\nHIDE-A-YEAR SUMMARY")
for row in summary:
    print(row)

print("\nERROR BY ACTIVITY LEVEL")
levels = error_by_level(summary)

for level, (mae, bias) in levels.items():
    print(
        level,
        "| mean absolute error:", round(mae, 2), "%",
        "| mean signed error:", round(bias, 2), "%"
    )

print("\nYears validated:", len(summary))
print("All years present:", sorted(r["year"] for r in summary))

Validation rows: 24
First result:
{'region': 0, 'preset': 'punjab', 'year': 2012, 'actual': array([    0.,    60.,    52.,  1769.,  3540.,    56.,    45.,    64.,
          71.,  9689., 12122.,   108.]), 'pred': array([    0.        ,    59.92231804,    57.49069822,  1742.24114253,
        3456.79055087,    53.04897975,    37.5657193 ,    67.50774154,
          70.63599726,  9622.77646462, 12141.52307497,   126.33010558]), 'pred_lo': array([    0.        ,    56.70612619,    52.7419366 ,  1662.59610813,
        3366.04377197,    49.29307584,    35.04716539,    62.81512474,
          65.50614999,  9439.47967487, 11955.17887257,   118.74700936]), 'pred_hi': array([    0.        ,    64.70937263,    60.65642385,  1823.44964446,
        3549.91246792,    56.88646003,    40.94527771,    71.94440537,
          75.974069  ,  9807.88669336, 12348.16130937,   134.40361403]), 'base_none': array([   0.,   34.,   30., 1004., 1986.,   29.,   22.,   36.,   40.,
       5538., 6969.,   74.]), 'base_gl

In [56]:
from pathlib import Path

os.chdir("/content/emberwise_m2_work")

for filename in [
    "docs/schema.md",
    "CHANGELOG.md"
]:
    path = Path(filename)
    print("\n" + "=" * 65)
    print(filename)
    print("=" * 65)

    if path.exists():
        print(path.read_text())
    else:
        print("File not found:", filename)


docs/schema.md
# EMBERWISE data contract (schema_version 1.0)

Generated from `pipeline/tables/contracts.py`. Do not edit by hand.
Owner: M3. Deputy: M6 (additive changes only while M3 sleeps, H16-20). Protocol: `CHANGELOG.md`.
Source: Build Plan v4, Section 5. Anything the plan does not state is marked [verify].

## Rules that apply to every file

- Every JSON file starts with `schema_version`, `generated_at`, `mock`. `mock: true` data is synthetic and must never be quoted as real.
- Every object is closed: a missing field or an extra field fails `check_contract`.
- **Missing data is a gap (`null`), never `0`.** Empty or failed files show up in `coverage`.
- **Never add MODIS and VIIRS.** One source of truth per day: VIIRS where it has data, otherwise MODIS x conversion ratio.
- Every number has a label: Measured, Estimated, Indicative, Provisional. Era A = Indicative, era B = Estimated, eras C and D = Measured; any month using NRT is Provisional.
- The headline value (`harm`) is rel

In [57]:
import os
import json
from pathlib import Path

os.chdir("/content/emberwise_m2_work")

# 1. Inspect existing generated JSON structures.
for filename in [
    "web/public/data/conversion.json",
    "web/public/data/validation.json"
]:
    path = Path(filename)

    print("\n" + "=" * 60)
    print(filename)
    print("=" * 60)

    if path.exists():
        data = json.loads(path.read_text())

        print("Top-level keys:", list(data.keys()))
        print("Mock:", data.get("mock"))

        for key, value in data.items():
            if isinstance(value, dict):
                print(f"\n{key} keys:", list(value.keys()))
                for subkey, subvalue in value.items():
                    if isinstance(subvalue, dict):
                        print(f"  {subkey} keys:", list(subvalue.keys()))
    else:
        print("File not found")

# 2. Inspect the schema contract definitions.
contract_path = Path("pipeline/tables/contracts.py")

print("\n" + "=" * 60)
print("CONTRACT DEFINITIONS")
print("=" * 60)

if contract_path.exists():
    lines = contract_path.read_text().splitlines()

    for i, line in enumerate(lines):
        if any(word in line.lower() for word in [
            "conversion", "validation", "hide_a_year",
            "error_by_level", "era_b_test", "ratio_trend",
            "nrt_seam"
        ]):
            start = max(0, i - 2)
            end = min(len(lines), i + 12)
            print(f"\n--- Lines {start + 1}-{end} ---")
            print("\n".join(
                f"{j + 1}: {lines[j]}"
                for j in range(start, end)
            ))
else:
    print("contracts.py not found")


web/public/data/conversion.json
Top-level keys: ['schema_version', 'generated_at', 'mock', 'unit', 'learning_window', 'k', 'zones', 'columns', 'ratio_by_year']
Mock: True

columns keys: ['zone', 'month', 'era', 'ratio', 'lo', 'hi', 'widened_lo', 'widened_hi', 'n_support']

ratio_by_year keys: ['columns']
  columns keys: ['zone', 'month', 'year', 'ratio', 'in_learning_window']

web/public/data/validation.json
Top-level keys: ['schema_version', 'generated_at', 'mock', 'hide_a_year', 'error_by_level', 'baselines', 'rank_test', 'seam_step', 'era_b_test', 'ratio_trend', 'res_sensitivity', 'nrt_seam', 'external_check', 'negative_control']
Mock: True

hide_a_year keys: ['owner', 'reviewer', 'status', 'label', 'result']
  result keys: ['columns']

error_by_level keys: ['owner', 'reviewer', 'status', 'label', 'result']

baselines keys: ['owner', 'reviewer', 'status', 'label', 'result']
  result keys: ['metric', 'n', 'per_zone_month', 'global_ratio', 'no_conversion', 'beats_both']

rank_test ke

In [58]:
import os
os.chdir("/content/emberwise_m2_work")

from pathlib import Path

path = Path("pipeline/tables/contracts.py")
lines = path.read_text().splitlines()

print("\n".join(
    f"{i+1}: {lines[i]}"
    for i in range(152, 178)
))

153:     "unit": string("What the ratio converts."),
154:     "learning_window": arr(integer(), "[first, last] year the ratios were learned on.", n=2),
155:     "k": num("Shrinkage pseudo tile-days."),
156:     "zones": arr(string(), "H3 res-3 zone ids; zone code = index."),
157:     "columns": obj({
158:         "zone": col(arr(integer()), "Zone code."),
159:         "month": col(arr(integer()), "Calendar month 1-12."),
160:         "era": col(arr(string(enum=["A", "B"])), "A = Terra only, B = Terra U Aqua."),
161:         "ratio": col(NNUM, "VIIRS tile-days per MODIS tile-day (shrunk toward the region ratio)."),
162:         "lo": col(NNUM, "10th percentile of yearly ratios."),
163:         "hi": col(NNUM, "90th percentile of yearly ratios."),
164:         "widened_lo": col(NNUM, "lo widened by tested error; used for eras A and B."),
165:         "widened_hi": col(NNUM, "hi widened by tested error; used for eras A and B."),
166:         "n_support": col(arr(integer()), "MODIS tile-da

In [59]:
from pathlib import Path

path = Path("pipeline/tables/contracts.py")
lines = path.read_text().splitlines()

print("\n".join(
    f"{i+1}: {lines[i]}"
    for i in range(460, 530)
))

461:     if name == "region_daily":
462:         cut = day_index(C.VIIRS_START)
463:         for pid, p in d["presets"].items():
464:             for k, v in p.items():
465:                 if isinstance(v, list) and len(v) != d["n_days"]:
466:                     e.append(f"presets.{pid}.{k}: length {len(v)} != n_days {d['n_days']}")
467:             for k in ("viirs_td", "viirs_hotspots"):
468:                 if any(v is not None for v in p[k][:cut]):
469:                     e.append(f"presets.{pid}.{k}: must be null before {C.VIIRS_START} (a gap, never zero)")
470:     if name == "tile_geometry":
471:         n = len(d["tiles"])
472:         for k in ("lng", "lat", "region", "zone"):
473:             if len(d[k]) != n:
474:                 e.append(f"{k}: length {len(d[k])} != tiles {n}")
475:         if len(d["verts"]) != 12 * n:
476:             e.append(f"verts: expected {12 * n} numbers, got {len(d['verts'])}")
477:         fb = d["fallback"]
478:         if len(fb["parent"]) 

In [60]:
%%writefile /content/emberwise_m2_work/pipeline/validate/checks.py
"""M2: tested widening, hide-a-year block, error by level, era-B test, ratio trend. Arrays: [region, zone, year, month]."""
import numpy as np
from scipy import stats
from pipeline.calibrate.ratios import fit_ratio, PRESETS, YEARS

PILOT = [9, 10]            # Oct, Nov as month indexes
MIN_ACTUAL = 10            # ignore zone-months with fewer than this many VIIRS tile-days (noise-dominated)
TREND_P, TREND_REL = 0.10, 2.0


def p80(errs):
    e = np.abs(np.asarray(errs, float))
    return float(np.percentile(e, 80)) if len(e) else None


def cell_errors(V, M, years=YEARS, k=20, months=PILOT, min_actual=MIN_ACTUAL):
    """Zone-month relative errors (pred-actual)/actual for the hidden year, Oct-Nov, learned without that year."""
    Vz, Mz = V.sum(0), M.sum(0)
    out = []
    for i in range(len(years)):
        keep = [j for j in range(len(years)) if j != i]
        r, _, _ = fit_ratio(Vz[:, keep], Mz[:, keep], k)
        for m in months:
            a, p = Vz[:, i, m], Mz[:, i, m] * r[:, m]
            ok = (a >= min_actual) & np.isfinite(p)
            out.extend(((p[ok] - a[ok]) / a[ok]).tolist())
    return np.array(out)


def block(result, mock, label="Measured", reviewer="M3"):
    return {"owner": "M2", "reviewer": reviewer, "status": "pending" if mock else "done",
            "label": label, "result": result}


def hide_a_year_block(rows, w, mock):
    """Contract columns: region (preset index), year, predicted, actual (all-month totals), in_range (widened by w)."""
    cols = {k: [] for k in ("region", "year", "predicted", "actual", "in_range")}
    for r in rows:
        pred, act = float(r["pred"].sum()), float(r["actual"].sum())
        lo, hi = float(r["pred_lo"].sum()) * (1 - w), float(r["pred_hi"].sum()) * (1 + w)
        cols["region"].append(int(r["region"])); cols["year"].append(int(r["year"]))
        cols["predicted"].append(round(pred, 1)); cols["actual"].append(round(act, 1))
        cols["in_range"].append(bool(lo <= act <= hi))
    return block({"columns": cols}, mock)


def error_by_level_block(table, w, mock, preset="punjab_haryana"):
    t = sorted(table, key=lambda r: r["actual"]); n = len(t)
    c1, c2 = round(n / 3), round(2 * n / 3)
    levels = {}
    for name, g in (("low", t[:c1]), ("mid", t[c1:c2]), ("high", t[c2:])):
        if g:
            e = np.array([r["err"] for r in g], float)
            levels[name] = {"years": [int(r["year"]) for r in g],
                            "mean_abs_error_pct": round(float(np.nanmean(np.abs(e))), 2),
                            "bias_pct": round(float(np.nanmean(e)), 2)}
    return block({"preset": preset, "months": [10, 11],
                  "metric": "percent error of Oct-Nov tile-days, hidden year, grouped by actual activity (3/2/3 years)",
                  "levels": levels, "widening_used": round(w, 4),
                  "note": "Eight years only; tail error is reported as found."}, mock)


def era_b_test(T, Mt, rA, rB, years, mock, months=PILOT, min_actual=MIN_ACTUAL):
    """2003-2011: does Terra x r_A agree with (Terra U Aqua) x r_B? Consistency check; no VIIRS exists in these years."""
    T, Mt = T.sum(0), Mt.sum(0)
    rel, by_year = [], []
    for yi, y in enumerate(years):
        sa = sb = 0.0
        for m in months:
            a, b = T[:, yi, m] * rA[:, m], Mt[:, yi, m] * rB[:, m]
            ok = (b >= min_actual) & np.isfinite(a) & np.isfinite(b)
            rel.extend(((a[ok] - b[ok]) / b[ok]).tolist()); sa += a[ok].sum(); sb += b[ok].sum()
        by_year.append({"year": int(y), "signed_pct": round(100 * (sa - sb) / sb, 2) if sb > 0 else None})
    rel = np.array(rel)
    e_b = p80(rel)
    res = {"years": [int(years[0]), int(years[-1])], "months": [10, 11],
           "metric": "|Terra x r_A - MODIS_ALL x r_B| / (MODIS_ALL x r_B), zone-month",
           "n_cells": int(len(rel)),
           "p80_abs_rel_diff": None if e_b is None else round(e_b, 4),
           "median_signed_rel_diff": None if not len(rel) else round(float(np.median(rel)), 4),
           "by_year": by_year,
           "note": "Consistency check between the two conversions; no VIIRS exists in these years, so it is not a test against truth."}
    return block(res, mock, label="Estimated"), e_b


def ratio_trend_block(V, M, years, mock, months=PILOT):
    ys = np.array(years)
    def one(vv, mm):
        y = np.where(mm > 0, vv / np.where(mm > 0, mm, 1), np.nan); ok = np.isfinite(y)
        if ok.sum() < 5:
            return None
        slope = stats.theilslopes(y[ok], ys[ok])[0]
        tau, p = stats.kendalltau(ys[ok], y[ok])
        rel = 100 * slope / np.median(y[ok])
        return dict(n_years=int(ok.sum()), median_ratio=round(float(np.median(y[ok])), 3),
                    slope_per_year=round(float(slope), 4), rel_slope_pct_per_year=round(float(rel), 2),
                    kendall_tau=round(float(tau), 3), p_value=round(float(p), 3),
                    clear_trend=bool(p < TREND_P and abs(rel) >= TREND_REL))
    rows = []
    for pid, parts in PRESETS.items():
        Vp, Mp = V[parts].sum(0).sum(0), M[parts].sum(0).sum(0)
        for m in months:
            r = one(Vp[:, m], Mp[:, m])
            rows.append({"preset": pid, "month": m + 1, **(r or {"n_years": 0, "clear_trend": False})})
    Vz, Mz = V.sum(0), M.sum(0)
    tested = flagged = 0
    for z in range(Vz.shape[0]):
        for m in months:
            r = one(Vz[z, :, m], Mz[z, :, m])
            if r:
                tested += 1; flagged += r["clear_trend"]
    res = {"method": "Theil-Sen slope and Kendall tau on yearly VIIRS/MODIS tile-day ratio, learning window, Oct and Nov",
           "rule": f"clear trend if Kendall p < {TREND_P} and |relative slope| >= {TREND_REL}% per year",
           "years": [int(y) for y in years], "rows": rows,
           "zones_tested": tested, "zones_with_clear_trend": int(flagged),
           "clear_trend": bool(any(r["clear_trend"] for r in rows)),
           "note": "Eight years give low power; zone-level flags are a screen, not proof."}
    return block(res, mock)

Writing /content/emberwise_m2_work/pipeline/validate/checks.py


In [62]:
import os
from pathlib import Path

WORK = Path("/content/emberwise_m2_work")

assert WORK.exists(), "M2 working directory not found."

os.chdir(WORK)

print("Working directory:", Path.cwd())
print("Coverage file exists:",
      Path("web/public/data/coverage.json").exists())
print("Tile-day file exists:",
      Path("data/processed/tile_day.parquet").exists())
print("M3 original directory exists:",
      Path("/content/emberwise").exists())

Working directory: /content/emberwise_m2_work
Coverage file exists: True
Tile-day file exists: True
M3 original directory exists: True


In [63]:
import json
from pathlib import Path

coverage_path = Path("web/public/data/coverage.json")

with coverage_path.open() as f:
    coverage = json.load(f)

print("Top-level keys:")
print(list(coverage.keys()))

print("\nCoverage structure:")

def inspect_structure(obj, level=0):
    indent = "  " * level

    if isinstance(obj, dict):
        for key, value in obj.items():
            print(f"{indent}{key}: {type(value).__name__}")

            if isinstance(value, (dict, list)):
                inspect_structure(value, level + 1)

            if level >= 3:
                break

    elif isinstance(obj, list):
        print(f"{indent}Number of entries: {len(obj)}")

        for item in obj[:3]:
            inspect_structure(item, level + 1)

inspect_structure(coverage)

Top-level keys:
['schema_version', 'generated_at', 'mock', 'epoch', 'sensor_sets', 'products', 'columns']

Coverage structure:
schema_version: str
generated_at: str
mock: bool
epoch: str
sensor_sets: list
  Number of entries: 5
products: list
  Number of entries: 2
columns: dict
  sensor_set: list
    Number of entries: 12
  product: list
    Number of entries: 12
  start: list
    Number of entries: 12
  end: list
    Number of entries: 12
  has_data: list
    Number of entries: 12


In [64]:
import json
from pathlib import Path

coverage = json.loads(
    Path("web/public/data/coverage.json").read_text()
)

print(json.dumps(coverage, indent=2)[:12000])

{
  "schema_version": "1.0",
  "generated_at": "2026-11-14T00:00:00Z",
  "mock": true,
  "epoch": "2000-11-01",
  "sensor_sets": [
    "TERRA",
    "MODIS_ALL",
    "VIIRS_SNPP",
    "VIIRS_NOAA20",
    "VIIRS_NOAA21"
  ],
  "products": [
    "SP",
    "NRT"
  ],
  "columns": {
    "sensor_set": [
      0,
      0,
      0,
      0,
      1,
      1,
      1,
      1,
      2,
      2,
      3,
      3
    ],
    "product": [
      0,
      0,
      0,
      1,
      0,
      0,
      0,
      1,
      0,
      1,
      0,
      1
    ],
    "start": [
      0,
      1712,
      1715,
      9435,
      0,
      1712,
      1715,
      9435,
      4097,
      9435,
      6270,
      9435
    ],
    "end": [
      1711,
      1714,
      9434,
      9508,
      1711,
      1714,
      9434,
      9508,
      9434,
      9497,
      9434,
      9508
    ],
    "has_data": [
      true,
      false,
      true,
      true,
      true,
      false,
      true,
      true,
      true,
      

In [65]:
import json
import calendar
from pathlib import Path
from datetime import date, timedelta

# Read coverage metadata
coverage = json.loads(
    Path("web/public/data/coverage.json").read_text()
)

epoch = date.fromisoformat(coverage["epoch"])

sensor_names = ["TERRA", "MODIS_ALL", "VIIRS_SNPP"]
product_name = "SP"

sensor_codes = {
    name: coverage["sensor_sets"].index(name)
    for name in sensor_names
}

product_code = coverage["products"].index(product_name)

cols = coverage["columns"]

# Build true-coverage intervals for each sensor
intervals = {sensor: [] for sensor in sensor_names}

for sensor, sensor_code in sensor_codes.items():
    for sc, pc, start, end, has_data in zip(
        cols["sensor_set"],
        cols["product"],
        cols["start"],
        cols["end"],
        cols["has_data"]
    ):
        if sc == sensor_code and pc == product_code and has_data:
            intervals[sensor].append((start, end))


# Check whether a day range is completely covered
def fully_covered(start_day, end_day, ranges):
    """
    Check inclusive day offsets.
    Multiple adjacent intervals may jointly cover a month.
    """
    relevant = sorted(
        (max(start_day, a), min(end_day, b))
        for a, b in ranges
        if b >= start_day and a <= end_day
    )

    next_day = start_day

    for start, end in relevant:
        if start > next_day:
            return False

        next_day = max(next_day, end + 1)

        if next_day > end_day:
            return True

    return next_day > end_day


# Check all months in the learning window
eligible_months = []
excluded_months = []

for year in range(2012, 2020):
    for month in range(1, 13):
        month_start = date(year, month, 1)
        month_end = date(year, month, calendar.monthrange(year, month)[1])

        start_day = (month_start - epoch).days
        end_day = (month_end - epoch).days

        sensor_checks = {
            sensor: fully_covered(
                start_day,
                end_day,
                intervals[sensor]
            )
            for sensor in sensor_names
        }

        if all(sensor_checks.values()):
            eligible_months.append((year, month))
        else:
            excluded_months.append((year, month, sensor_checks))


print("Sensors checked:", sensor_names)
print("Product:", product_name)
print("Learning window: 2012–2019")

print("\nEligible months:", len(eligible_months))
print("Excluded months:", len(excluded_months))

print("\nExcluded months:")
for year, month, checks in excluded_months:
    print(f"{year}-{month:02d}: {checks}")

print("\nFirst eligible months:", eligible_months[:5])
print("Last eligible months:", eligible_months[-5:])

Sensors checked: ['TERRA', 'MODIS_ALL', 'VIIRS_SNPP']
Product: SP
Learning window: 2012–2019

Eligible months: 95
Excluded months: 1

Excluded months:
2012-01: {'TERRA': True, 'MODIS_ALL': True, 'VIIRS_SNPP': False}

First eligible months: [(2012, 2), (2012, 3), (2012, 4), (2012, 5), (2012, 6)]
Last eligible months: [(2019, 8), (2019, 9), (2019, 10), (2019, 11), (2019, 12)]


In [66]:
import os
import inspect

os.chdir("/content/emberwise_m2_work")

from pipeline.calibrate import ratios

print("Function location:")
print(inspect.getfile(ratios.build_cubes))

print("\nCurrent build_cubes() code:")
print(inspect.getsource(ratios.build_cubes))

Function location:
/content/pipeline/calibrate/ratios.py

Current build_cubes() code:
def build_cubes(
    path,
    sensors=("VIIRS_SNPP", "MODIS_ALL", "TERRA"),
    years=YEARS
):
    """Build [region, zone, year, month] arrays from tile_day.parquet."""

    import polars as pl

    td = pl.read_parquet(path)

    zones = sorted(
        td.filter(
            pl.col("region").is_in(BASE_REGIONS)
        )["zone_r3"].unique().to_list()
    )

    zone_index = {z: i for i, z in enumerate(zones)}

    cubes = {
        sensor: np.zeros(
            (len(BASE_REGIONS), len(zones), len(years), 12),
            dtype=float
        )
        for sensor in sensors
    }

    grouped = (
        td.filter(
            pl.col("region").is_in(BASE_REGIONS)
            & (pl.col("product") == "SP")
            & pl.col("sensor_set").is_in(list(sensors))
            & pl.col("date").dt.year().is_between(
                years[0], years[-1]
            )
        )
        .group_by(
            "

In [67]:
import os
import sys
import importlib
from pathlib import Path

WORK = Path("/content/emberwise_m2_work").resolve()

os.chdir(WORK)

target_file = WORK / "pipeline" / "calibrate" / "ratios.py"

assert target_file.exists(), (
    f"M2 working-copy module not found: {target_file}"
)

# Put the working copy first on Python's import path.
work_path = str(WORK)

if work_path in sys.path:
    sys.path.remove(work_path)

sys.path.insert(0, work_path)

# Remove previously imported pipeline modules so Python reloads
# them from the working copy.
for name in list(sys.modules):
    if name == "pipeline" or name.startswith("pipeline."):
        del sys.modules[name]

from pipeline.calibrate import ratios

print("Current directory:", Path.cwd())
print("Expected module:", target_file)
print("Actually imported:", Path(ratios.__file__).resolve())

assert Path(ratios.__file__).resolve() == target_file

print("\nPASS: Python is using the M2 working copy.")

Current directory: /content/emberwise_m2_work
Expected module: /content/emberwise_m2_work/pipeline/calibrate/ratios.py
Actually imported: /content/emberwise_m2_work/pipeline/calibrate/ratios.py

PASS: Python is using the M2 working copy.


In [68]:
import inspect
from pipeline.calibrate import ratios

print("========== fit_ratio() ==========")
print(inspect.getsource(ratios.fit_ratio))

print("\n========== ratio_range() ==========")
print(inspect.getsource(ratios.ratio_range))

========== fit_ratio() ==========
def fit_ratio(V, M, k=K):
    """Fit shrinkage ratios.

    V and M shapes: [zone, year, month].
    """

    V = np.asarray(V, dtype=float)
    M = np.asarray(M, dtype=float)

    if V.shape != M.shape or V.ndim != 3:
        raise ValueError("V and M must have matching [zone, year, month] shapes.")

    if k < 0:
        raise ValueError("k must be non-negative.")

    v_region = V.sum(axis=(0, 1))
    m_region = M.sum(axis=(0, 1))

    r_region = np.divide(
        v_region,
        m_region,
        out=np.full_like(v_region, np.nan),
        where=m_region > 0
    )

    # Use zero contribution for months without regional observations.
    prior = np.nan_to_num(r_region, nan=0.0)

    r_hat = np.divide(
        V.sum(axis=1) + k * prior,
        M.sum(axis=1) + k,
        out=np.broadcast_to(prior, (V.shape[0], V.shape[2])).copy(),
        where=(M.sum(axis=1) + k) > 0
    )

    yearly = np.divide(
        V + k * prior[None, None, :],
        M 

In [69]:
import inspect
from pipeline.validate import hide_a_year

print("========== hide_a_year() ==========")
print(inspect.getsource(hide_a_year.hide_a_year))

print("\n========== summarize() ==========")
print(inspect.getsource(hide_a_year.summarize))

print("\n========== error_by_level() ==========")
print(inspect.getsource(hide_a_year.error_by_level))

========== hide_a_year() ==========
def hide_a_year(V, M, years=YEARS, k=20):
    """Learn on all but one year and predict the hidden year."""

    V = np.asarray(V, dtype=float)
    M = np.asarray(M, dtype=float)

    if V.shape != M.shape or V.ndim != 4:
        raise ValueError(
            "V and M must have matching [region, zone, year, month] shapes."
        )

    if V.shape[2] != len(years):
        raise ValueError("Number of years does not match the data.")

    rows = []

    Vz = V.sum(axis=0)
    Mz = M.sum(axis=0)

    for i, year in enumerate(years):

        keep = [j for j in range(len(years)) if j != i]

        r_hat, yearly, r_region = fit_ratio(
            Vz[:, keep],
            Mz[:, keep],
            k
        )

        lo, hi = ratio_range(r_hat, yearly)

        total_v = Vz[:, keep].sum()
        total_m = Mz[:, keep].sum()

        global_ratio = (
            total_v / total_m if total_m > 0 else np.nan
        )

        for pi, (preset, parts) in enu

In [70]:
import inspect
from pipeline.calibrate import ratios
from pipeline.validate import hide_a_year as validation_module

print("========== ratios.py imports and constants ==========")
print("\n".join(inspect.getsource(ratios).splitlines()[:35]))

print("\n========== hide_a_year.py imports and constants ==========")
print("\n".join(inspect.getsource(validation_module).splitlines()[:45]))

print("\n========== Current function references ==========")
print("Validation fit_ratio is ratios.fit_ratio:",
      validation_module.fit_ratio is ratios.fit_ratio)
print("Validation ratio_range is ratios.ratio_range:",
      validation_module.ratio_range is ratios.ratio_range)

========== ratios.py imports and constants ==========

"""M2: MODIS to VIIRS conversion ratio calibration."""

import numpy as np

K = 20

YEARS = list(range(2012, 2020))

BASE_REGIONS = ["punjab", "haryana"]

PRESETS = {
    "punjab": [0],
    "haryana": [1],
    "punjab_haryana": [0, 1]
}



def build_cubes(
    path,
    coverage_path="web/public/data/coverage.json",
    sensors=("VIIRS_SNPP", "MODIS_ALL", "TERRA"),
    years=YEARS
):
    """Build coverage-aware [region, zone, year, month] arrays."""

    import json
    import polars as pl
    from datetime import date, timedelta

    td = pl.read_parquet(path)

    with open(coverage_path) as f:
        coverage = json.load(f)

========== hide_a_year.py imports and constants ==========

"""M2: Hide-a-year validation."""

import numpy as np

from pipeline.calibrate.ratios import (
    fit_ratio,
    ratio_range,
    PRESETS,
    YEARS
)

OCT_NOV = [9, 10]


def hide_a_year(V, M, years=YEARS, k=20):
    """Learn on all but one year 

In [71]:
import inspect
from pipeline.calibrate import ratios

print("========== M2 build_cubes() ==========")
print(inspect.getsource(ratios.build_cubes))

print("\n========== M2 fit_ratio() ==========")
print(inspect.getsource(ratios.fit_ratio))

print("\n========== M2 ratio_range() ==========")
print(inspect.getsource(ratios.ratio_range))

========== M2 build_cubes() ==========
def build_cubes(
    path,
    coverage_path="web/public/data/coverage.json",
    sensors=("VIIRS_SNPP", "MODIS_ALL", "TERRA"),
    years=YEARS
):
    """Build coverage-aware [region, zone, year, month] arrays."""

    import json
    import polars as pl
    from datetime import date, timedelta

    td = pl.read_parquet(path)

    with open(coverage_path) as f:
        coverage = json.load(f)

    epoch = date.fromisoformat(coverage["epoch"])
    sensor_names = coverage["sensor_sets"]
    product_names = coverage["products"]
    c = coverage["columns"]

    valid_intervals = {}

    for sensor in sensors:
        si = sensor_names.index(sensor)
        pi = product_names.index("SP")

        intervals = []

        for i in range(len(c["sensor_set"])):
            if (
                c["sensor_set"][i] == si
                and c["product"][i] == pi
                and c["has_data"][i]
            ):
                intervals.append((
           

In [72]:
import json
import numpy as np
import polars as pl

from pathlib import Path
from datetime import date, timedelta

from pipeline.calibrate import ratios

def build_cubes(
    path,
    coverage_path="web/public/data/coverage.json",
    sensors=("VIIRS_SNPP", "MODIS_ALL", "TERRA"),
    years=ratios.YEARS
):
    """Build common-coverage [region, zone, year, month] cubes.

    Months are retained only when every requested sensor has
    complete SP coverage for the entire calendar month.

    Uncovered months are NaN, not zero.
    """

    td = pl.read_parquet(path)

    with open(coverage_path) as f:
        coverage = json.load(f)

    epoch = date.fromisoformat(coverage["epoch"])
    sensor_names = coverage["sensor_sets"]
    product_names = coverage["products"]
    columns = coverage["columns"]

    if "SP" not in product_names:
        raise ValueError("SP product is missing from coverage metadata.")

    pi = product_names.index("SP")

    # Build valid coverage intervals for each requested sensor.
    valid_intervals = {}

    for sensor in sensors:
        if sensor not in sensor_names:
            raise ValueError(f"Sensor missing from coverage metadata: {sensor}")

        si = sensor_names.index(sensor)
        intervals = []

        for i in range(len(columns["sensor_set"])):
            if (
                columns["sensor_set"][i] == si
                and columns["product"][i] == pi
                and columns["has_data"][i]
            ):
                intervals.append((
                    epoch + timedelta(days=columns["start"][i]),
                    epoch + timedelta(days=columns["end"][i])
                ))

        # Merge overlapping or adjacent intervals.
        intervals.sort()
        merged = []

        for start, end in intervals:
            if not merged or start > merged[-1][1] + timedelta(days=1):
                merged.append([start, end])
            else:
                merged[-1][1] = max(merged[-1][1], end)

        valid_intervals[sensor] = [
            (start, end) for start, end in merged
        ]

    def fully_covered(sensor, month_start, month_end):
        return any(
            start <= month_start and end >= month_end
            for start, end in valid_intervals[sensor]
        )

    # Common month mask: every requested sensor must cover the month.
    common_months = {}

    for year in years:
        for month in range(1, 13):
            month_start = date(year, month, 1)

            if month == 12:
                month_end = date(year + 1, 1, 1) - timedelta(days=1)
            else:
                month_end = date(year, month + 1, 1) - timedelta(days=1)

            common_months[(year, month)] = all(
                fully_covered(sensor, month_start, month_end)
                for sensor in sensors
            )

    zones = sorted(
        td.filter(
            pl.col("region").is_in(ratios.BASE_REGIONS)
        )["zone_r3"].unique().to_list()
    )

    zone_index = {zone: i for i, zone in enumerate(zones)}

    shape = (
        len(ratios.BASE_REGIONS),
        len(zones),
        len(years),
        12
    )

    # NaN means the month is not eligible for calibration.
    cubes = {
        sensor: np.full(shape, np.nan, dtype=float)
        for sensor in sensors
    }

    # Covered months begin at zero. Observed tile-day totals are
    # subsequently added into their corresponding cells.
    for sensor in sensors:
        for (year, month), eligible in common_months.items():
            if eligible:
                yi = years.index(year)
                cubes[sensor][:, :, yi, month - 1] = 0.0

    grouped = (
        td.filter(
            pl.col("region").is_in(ratios.BASE_REGIONS)
            & (pl.col("product") == "SP")
            & pl.col("sensor_set").is_in(list(sensors))
            & pl.col("date").dt.year().is_between(years[0], years[-1])
        )
        .group_by(
            "region",
            "zone_r3",
            "sensor_set",
            pl.col("date").dt.year().alias("year"),
            pl.col("date").dt.month().alias("month")
        )
        .agg(
            pl.col("n_tile_day").sum().alias("total")
        )
    )

    for region, zone, sensor, year, month, total in grouped.iter_rows():
        if not common_months.get((year, month), False):
            continue

        ri = ratios.BASE_REGIONS.index(region)
        zi = zone_index[zone]
        yi = years.index(year)

        cubes[sensor][ri, zi, yi, month - 1] = float(total)

    return cubes, zones


ratios.build_cubes = build_cubes

print("Updated build_cubes() in the current notebook session.")
print("Working directory:", Path.cwd())
print("Function module:", ratios.__file__)

Updated build_cubes() in the current notebook session.
Working directory: /content/emberwise_m2_work
Function module: /content/emberwise_m2_work/pipeline/calibrate/ratios.py


In [73]:
from pathlib import Path
import numpy as np

WORK = Path("/content/emberwise_m2_work")
os.chdir(WORK)

cubes, zones = ratios.build_cubes(
    path="data/processed/tile_day.parquet",
    coverage_path="web/public/data/coverage.json"
)

print("Zones:", len(zones))
print("Sensors:", list(cubes))

for sensor, cube in cubes.items():
    valid_months = np.isfinite(cube[0, 0]).sum()
    missing_months = np.isnan(cube[0, 0]).sum()

    print(
        sensor,
        "shape:", cube.shape,
        "valid months per zone:", valid_months,
        "missing months per zone:", missing_months,
        "total:", np.nansum(cube)
    )

# Check that all sensors have exactly the same missing-month mask.
masks = {
    sensor: np.isfinite(cube)
    for sensor, cube in cubes.items()
}

reference = masks[list(cubes)[0]]

for sensor, mask in masks.items():
    print(
        sensor,
        "same coverage mask:",
        np.array_equal(reference, mask)
    )

assert all(
    np.array_equal(reference, mask)
    for mask in masks.values()
), "ERROR: sensor coverage masks differ."

print("\nPASS: all sensors have identical month eligibility.")

Zones: 16
Sensors: ['VIIRS_SNPP', 'MODIS_ALL', 'TERRA']
VIIRS_SNPP shape: (2, 16, 8, 12) valid months per zone: 95 missing months per zone: 1 total: 341629.0
MODIS_ALL shape: (2, 16, 8, 12) valid months per zone: 95 missing months per zone: 1 total: 196070.0
TERRA shape: (2, 16, 8, 12) valid months per zone: 95 missing months per zone: 1 total: 108083.0
VIIRS_SNPP same coverage mask: True
MODIS_ALL same coverage mask: True
TERRA same coverage mask: True

PASS: all sensors have identical month eligibility.


In [74]:
import polars as pl

td = pl.read_parquet("data/processed/tile_day.parquet")

print("January 2012 rows by sensor:")
print(
    td.filter(
        (pl.col("product") == "SP")
        & (pl.col("date").dt.year() == 2012)
        & (pl.col("date").dt.month() == 1)
        & pl.col("sensor_set").is_in(
            ["VIIRS_SNPP", "MODIS_ALL", "TERRA"]
        )
    )
    .group_by("sensor_set")
    .agg(
        pl.len().alias("rows"),
        pl.col("n_tile_day").sum().alias("total_n_tile_day")
    )
    .sort("sensor_set")
)

print("\nNull checks:")
print(
    td.select(
        pl.col("n_tile_day").null_count().alias("null_n_tile_day"),
        pl.col("n_tile_day").min().alias("min_n_tile_day"),
        pl.col("n_tile_day").max().alias("max_n_tile_day")
    )
)

January 2012 rows by sensor:
shape: (3, 3)
┌────────────┬──────┬──────────────────┐
│ sensor_set ┆ rows ┆ total_n_tile_day │
│ ---        ┆ ---  ┆ ---              │
│ str        ┆ u32  ┆ i32              │
╞════════════╪══════╪══════════════════╡
│ MODIS_ALL  ┆ 40   ┆ 40               │
│ TERRA      ┆ 18   ┆ 18               │
│ VIIRS_SNPP ┆ 26   ┆ 26               │
└────────────┴──────┴──────────────────┘

Null checks:
shape: (1, 3)
┌─────────────────┬────────────────┬────────────────┐
│ null_n_tile_day ┆ min_n_tile_day ┆ max_n_tile_day │
│ ---             ┆ ---            ┆ ---            │
│ u32             ┆ i32            ┆ i32            │
╞═════════════════╪════════════════╪════════════════╡
│ 0               ┆ 1              ┆ 1              │
└─────────────────┴────────────────┴────────────────┘


In [75]:
import inspect
from pipeline.tables import make_mocks

print("Generator location:", inspect.getfile(make_mocks))
print("\nRelevant generator functions:")
for name, obj in inspect.getmembers(make_mocks, inspect.isfunction):
    if any(word in name.lower() for word in ["tile", "day", "mock"]):
        print(f"\n--- {name} ---")
        print(inspect.getsource(obj))

Generator location: /content/emberwise_m2_work/pipeline/tables/make_mocks.py

Relevant generator functions:


In [76]:
from pathlib import Path

p = Path("pipeline/tables/make_mocks.py")

print("File:", p.resolve())
print("File length:", len(p.read_text().splitlines()), "lines")
print(p.read_text())

File: /content/emberwise_m2_work/pipeline/tables/make_mocks.py
File length: 6 lines
"""Write mock stand-ins for every upstream hand-off.  Usage: python -m pipeline.tables.make_mocks"""
from . import mocks

if __name__ == "__main__":
    mocks.generate()
    print("wrote mock hand-offs: data/processed/*.parquet and upstream JSON in web/public/data")



In [77]:
import inspect
from pipeline.tables import mocks

print("Generator file:", inspect.getfile(mocks))
print("\nFull generator source:\n")
print(inspect.getsource(mocks))

Generator file: /content/emberwise_m2_work/pipeline/tables/mocks.py

Full generator source:

"""Mock stand-ins for everything M3 receives from M1, M2, M5 and M6.

Writes data/processed/{tiles,tile_day}.parquet and the upstream JSON contracts
(regions, coverage, hourly, drift, conversion, validation, relay_sim, hotspot CSV
headers). M3's own tables are then built from these by build.py, exactly as they
will be from the real hand-offs.

Everything is "mock": true. Behaviour is meant to be realistic (Oct-Nov peak,
smaller Apr-May peak, quiet months, saturated days, VIIRS absent before 2012,
a jump in raw counts at 2012, NRT newest months, a data gap, a low-fire control
district). The VALUES are invented and must never be quoted.
"""
from __future__ import annotations

from datetime import date

import h3
import numpy as np
import polars as pl

from . import config as C
from .jsonio import envelope, qlist, write_json
from .timeutil import day_index, index_day, parse

SEED = 20261114
GENERA

In [78]:
import numpy as np
from pipeline.calibrate import ratios

def fit_ratio(V, M, k=ratios.K):
    """Fit shrinkage ratios while excluding missing month observations.

    V and M shapes: [zone, year, month].
    A month is usable only where both arrays have finite values.
    """

    V = np.asarray(V, dtype=float)
    M = np.asarray(M, dtype=float)

    if V.shape != M.shape or V.ndim != 3:
        raise ValueError(
            "V and M must have matching [zone, year, month] shapes."
        )

    if k < 0:
        raise ValueError("k must be non-negative.")

    valid = np.isfinite(V) & np.isfinite(M)

    V_valid = np.where(valid, V, 0.0)
    M_valid = np.where(valid, M, 0.0)

    # Count valid year-zone observations for each month.
    n_valid = valid.sum(axis=(0, 1))

    v_region = V_valid.sum(axis=(0, 1))
    m_region = M_valid.sum(axis=(0, 1))

    r_region = np.divide(
        v_region,
        m_region,
        out=np.full(V.shape[2], np.nan),
        where=(m_region > 0) & (n_valid > 0)
    )

    prior = np.nan_to_num(r_region, nan=0.0)

    v_zone = V_valid.sum(axis=1)
    m_zone = M_valid.sum(axis=1)

    r_hat = np.divide(
        v_zone + k * prior[None, :],
        m_zone + k,
        out=np.full((V.shape[0], V.shape[2]), np.nan),
        where=(m_zone + k) > 0
    )

    yearly = np.divide(
        V_valid + k * prior[None, None, :],
        M_valid + k,
        out=np.full(V.shape, np.nan),
        where=valid & ((M_valid + k) > 0)
    )

    # Months without any valid observations must remain missing.
    no_observations = n_valid == 0
    r_hat[:, no_observations] = np.nan
    yearly[:, :, no_observations] = np.nan

    return r_hat, yearly, r_region


def ratio_range(r_hat, yearly):
    """Calculate 10th–90th percentile ranges, ignoring missing years."""

    r_hat = np.asarray(r_hat, dtype=float)
    yearly = np.asarray(yearly, dtype=float)

    if yearly.ndim != 3 or r_hat.shape != (
        yearly.shape[0], yearly.shape[2]
    ):
        raise ValueError("Unexpected r_hat/yearly shapes.")

    with np.errstate(invalid="ignore"):
        lo = np.nanpercentile(yearly, 10, axis=1)
        hi = np.nanpercentile(yearly, 90, axis=1)

    lo = np.where(np.isfinite(r_hat), np.fmin(lo, r_hat), np.nan)
    hi = np.where(np.isfinite(r_hat), np.fmax(hi, r_hat), np.nan)

    return lo, hi


ratios.fit_ratio = fit_ratio
ratios.ratio_range = ratio_range

print("Updated fit_ratio() and ratio_range().")

Updated fit_ratio() and ratio_range().


In [81]:
# Small controlled test: one month is missing for every year.
V_test = np.array([[[20.0, np.nan], [40.0, np.nan]]])
M_test = np.array([[[10.0, np.nan], [20.0, np.nan]]])

r_hat_test, yearly_test, r_region_test = ratios.fit_ratio(
    V_test, M_test, k=20
)

lo_test, hi_test = ratios.ratio_range(
    r_hat_test, yearly_test
)

print("Regional ratio:", r_region_test)
print("Fitted ratio:", r_hat_test)
print("Yearly ratios:", yearly_test)
print("Lower range:", lo_test)
print("Upper range:", hi_test)

assert np.isclose(r_region_test[0], 2.0)
assert np.isclose(r_hat_test[0, 0], 2.0)
assert np.isnan(r_region_test[1])
assert np.isnan(r_hat_test[0, 1])
assert np.isnan(lo_test[0, 1])
assert np.isnan(hi_test[0, 1])

print("\nPASS: missing months remain missing; valid ratio is 2.0.")

Regional ratio: [ 2. nan]
Fitted ratio: [[ 2. nan]]
Yearly ratios: [[[ 2. nan]
  [ 2. nan]]]
Lower range: [[ 2. nan]]
Upper range: [[ 2. nan]]

PASS: missing months remain missing; valid ratio is 2.0.


In [80]:
import numpy as np
from pipeline.validate import hide_a_year as validation

def hide_a_year(V, M, years=validation.YEARS, k=20):
    """Hide one year and validate using common valid observations."""

    V = np.asarray(V, dtype=float)
    M = np.asarray(M, dtype=float)

    if V.shape != M.shape or V.ndim != 4:
        raise ValueError("V and M must have matching 4D shapes.")

    if V.shape[2] != len(years):
        raise ValueError("Number of years does not match the data.")

    rows = []

    valid = np.isfinite(V) & np.isfinite(M)
    Vz = np.where(valid, V, 0.0).sum(axis=0)
    Mz = np.where(valid, M, 0.0).sum(axis=0)

    for i, year in enumerate(years):
        keep = [j for j in range(len(years)) if j != i]

        r_hat, yearly, _ = validation.fit_ratio(
            Vz[:, keep], Mz[:, keep], k
        )
        lo, hi = validation.ratio_range(r_hat, yearly)

        total_v = Vz[:, keep].sum()
        total_m = Mz[:, keep].sum()
        global_ratio = total_v / total_m if total_m > 0 else np.nan

        for pi, (preset, parts) in enumerate(validation.PRESETS.items()):
            m = np.where(
                np.isfinite(M[parts]),
                M[parts], 0.0
            ).sum(axis=0)[:, i]

            actual_values = V[parts, :, i, :]
            actual = np.nansum(actual_values, axis=(0, 1))

            pred = np.nansum(m * r_hat, axis=0)
            pred_lo = np.nansum(m * lo, axis=0)
            pred_hi = np.nansum(m * hi, axis=0)

            rows.append({
                "region": pi,
                "preset": preset,
                "year": year,
                "actual": actual,
                "pred": pred,
                "pred_lo": pred_lo,
                "pred_hi": pred_hi,
                "base_none": m.sum(axis=0),
                "base_global": m.sum(axis=0) * global_ratio
            })

    return rows


validation.hide_a_year = hide_a_year
print("Updated hide_a_year().")

Updated hide_a_year().


In [82]:
import numpy as np
from pipeline.calibrate import ratios
from pipeline.validate import hide_a_year as validation

# Build cubes using common coverage.
cubes, zones = ratios.build_cubes(
    "data/processed/tile_day.parquet",
    "web/public/data/coverage.json"
)

V = cubes["VIIRS_SNPP"].sum(axis=0)
M = cubes["MODIS_ALL"].sum(axis=0)

# Fit conversion ratios.
r_hat, yearly, r_region = ratios.fit_ratio(V, M)
lo, hi = ratios.ratio_range(r_hat, yearly)

# Hold out each year.
rows = validation.hide_a_year(
    cubes["VIIRS_SNPP"],
    cubes["MODIS_ALL"]
)

summary = validation.summarize(
    rows,
    months=[9, 10],
    preset="punjab_haryana"
)

print("Zones:", len(zones))
print("Cube shape:", cubes["VIIRS_SNPP"].shape)
print("Finite fitted ratios:", np.isfinite(r_hat).sum())
print("Missing fitted ratios:", np.isnan(r_hat).sum())
print("Ratio percentiles:",
      np.nanpercentile(r_hat, [10, 50, 90]).round(3))

print("\nHoldout rows:", len(rows))
print("Punjab + Haryana October-November results:")
for row in summary:
    print(row)

print("\nPASS: calibration and holdout calculations completed.")

Zones: 16
Cube shape: (2, 16, 8, 12)
Finite fitted ratios: 192
Missing fitted ratios: 0
Ratio percentiles: [1.645 1.747 1.888]

Holdout rows: 24
Punjab + Haryana October-November results:
{'year': 2012, 'actual': np.float64(27262.0), 'pred': np.float64(27016.26399667042), 'err': np.float64(-0.901386557587781), 'in_range': True, 'err_none': np.float64(-43.063605017973735), 'err_global': np.float64(-0.899767232861816)}
{'year': 2013, 'actual': np.float64(36182.0), 'pred': np.float64(35915.27902074717), 'err': np.float64(-0.7371648312775141), 'in_range': True, 'err_none': np.float64(-42.9744071637831), 'err_global': np.float64(-0.6708280018381654)}
{'year': 2014, 'actual': np.float64(40445.0), 'pred': np.float64(40474.41231936551), 'err': np.float64(0.07272176873658173), 'in_range': True, 'err_none': np.float64(-42.57139325009272), 'err_global': np.float64(0.04701747807552082)}
{'year': 2015, 'actual': np.float64(27532.0), 'pred': np.float64(27465.075708229942), 'err': np.float64(-0.24307

In [83]:
import numpy as np
from pipeline.calibrate import ratios

V = cubes["VIIRS_SNPP"].sum(axis=0)
M_A = cubes["TERRA"].sum(axis=0)
M_B = cubes["MODIS_ALL"].sum(axis=0)

era_results = {}

for era, M in [("A", M_A), ("B", M_B)]:
    r_hat, yearly, r_region = ratios.fit_ratio(V, M)
    lo, hi = ratios.ratio_range(r_hat, yearly)

    era_results[era] = {
        "ratio": r_hat,
        "yearly": yearly,
        "lo": lo,
        "hi": hi,
        "n_support": np.isfinite(yearly).sum(axis=1)
    }

    print(f"\nERA {era}")
    print("Finite ratios:", np.isfinite(r_hat).sum())
    print("Ratio percentiles:",
          np.nanpercentile(r_hat, [10, 50, 90]).round(3))
    print("Support range:",
          int(era_results[era]["n_support"].min()),
          "to",
          int(era_results[era]["n_support"].max()))

print("\nPASS: Era A and Era B fitted independently.")


ERA A
Finite ratios: 192
Ratio percentiles: [2.901 3.162 3.543]
Support range: 7 to 8

ERA B
Finite ratios: 192
Ratio percentiles: [1.645 1.747 1.888]
Support range: 7 to 8

PASS: Era A and Era B fitted independently.


In [84]:
from pipeline.tables import contracts
import inspect

print("Contract functions:")
for name in dir(contracts):
    if "conversion" in name.lower() or "validate" in name.lower():
        obj = getattr(contracts, name)
        if callable(obj):
            print(f"\n{name}")
            try:
                print(inspect.signature(obj))
                print(inspect.getsource(obj)[:2500])
            except Exception:
                pass

Contract functions:

validate_file
(name: 'str', path: 'Path') -> 'list[str]'
def validate_file(name: str, path: Path) -> list[str]:
    """Validate one JSON file against contract `name`. Returns error strings."""
    text = Path(path).read_text(encoding="utf-8")
    errors = [f"banned phrase present: {b}" for b in find_banned(text)]
    try:
        data = json.loads(text)
    except json.JSONDecodeError as ex:
        return errors + [f"not valid JSON: {ex}"]
    found = sorted(Draft202012Validator(CONTRACTS[name][0]).iter_errors(data), key=lambda x: str(list(x.path)))
    for err in found[:20]:
        errors.append(f"{'.'.join(str(p) for p in err.path) or '<root>'}: {err.message[:200]}")
    if not errors:
        _columns_equal_length(data, "", errors)
        errors += _extra_checks(name, data)
    return errors


validate_hotspots_csv
(path: 'Path') -> 'list[str]'
def validate_hotspots_csv(path: Path) -> list[str]:
    """A hotspot CSV must start with exactly the contract header

In [85]:
from pipeline.tables.contracts import CONTRACTS
import json

schema, pattern = CONTRACTS["conversion"]

print("Expected filename pattern:", pattern)
print(json.dumps(schema, indent=2))

Expected filename pattern: conversion.json
{
  "type": "object",
  "properties": {
    "schema_version": {
      "const": "1.0",
      "description": "Contract version; bumped on any schema change."
    },
    "generated_at": {
      "type": "string",
      "description": "UTC build time, ISO 8601."
    },
    "mock": {
      "type": "boolean",
      "description": "true = synthetic mock data, never to be shown or quoted as real."
    },
    "unit": {
      "type": "string",
      "description": "What the ratio converts."
    },
    "learning_window": {
      "type": "array",
      "items": {
        "type": "integer"
      },
      "minItems": 2,
      "maxItems": 2,
      "description": "[first, last] year the ratios were learned on."
    },
    "k": {
      "type": "number",
      "description": "Shrinkage pseudo tile-days."
    },
    "zones": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "H3 res-3 zone ids; zone code = index."
   

In [86]:
from pathlib import Path

files = [
    "pipeline/calibrate/ratios.py",
    "pipeline/validate/hide_a_year.py",
    "pipeline/tables/contracts.py",
]

for name in files:
    p = Path(name)
    print(f"\n{'='*20} {name} {'='*20}")
    if p.exists():
        print(p.read_text(encoding="utf-8")[:14000])
    else:
        print("FILE NOT FOUND")


==================== pipeline/calibrate/ratios.py ====================

"""M2: MODIS to VIIRS conversion ratio calibration."""

import numpy as np

K = 20

YEARS = list(range(2012, 2020))

BASE_REGIONS = ["punjab", "haryana"]

PRESETS = {
    "punjab": [0],
    "haryana": [1],
    "punjab_haryana": [0, 1]
}



def build_cubes(
    path,
    coverage_path="web/public/data/coverage.json",
    sensors=("VIIRS_SNPP", "MODIS_ALL", "TERRA"),
    years=YEARS
):
    """Build coverage-aware [region, zone, year, month] arrays."""

    import json
    import polars as pl
    from datetime import date, timedelta

    td = pl.read_parquet(path)

    with open(coverage_path) as f:
        coverage = json.load(f)

    epoch = date.fromisoformat(coverage["epoch"])
    sensor_names = coverage["sensor_sets"]
    product_names = coverage["products"]
    c = coverage["columns"]

    valid_intervals = {}

    for sensor in sensors:
        si = sensor_names.index(sensor)
        pi = product_names.index("

In [87]:
import json
import numpy as np
from datetime import datetime, timezone
from pathlib import Path
from pipeline.calibrate import ratios
from pipeline.tables.contracts import validate_file

OUT = Path("web/public/data/conversion.json")
OUT.parent.mkdir(parents=True, exist_ok=True)

# Preserve missing coverage when combining the two regions.
def combine_regions(cube):
    valid = np.isfinite(cube)
    result = np.where(valid, cube, 0.0).sum(axis=0)
    result[~valid.any(axis=0)] = np.nan
    return result

V = combine_regions(cubes["VIIRS_SNPP"])
M_A = combine_regions(cubes["TERRA"])
M_B = combine_regions(cubes["MODIS_ALL"])

# Refit both eras using the corrected ratio functions.
era_data = {}
for era, M in [("A", M_A), ("B", M_B)]:
    ratio, yearly, _ = ratios.fit_ratio(V, M)
    lo, hi = ratios.ratio_range(ratio, yearly)

    era_data[era] = {
        "ratio": ratio,
        "lo": lo,
        "hi": hi,
        "support": np.nansum(M, axis=1)
    }

# Main zone-month table.
cols = {
    "zone": [], "month": [], "era": [],
    "ratio": [], "lo": [], "hi": [],
    "widened_lo": [], "widened_hi": [],
    "n_support": []
}

def json_number(value):
    return float(value) if np.isfinite(value) else None

for era in ["A", "B"]:
    d = era_data[era]
    for zi in range(len(zones)):
        for month in range(12):
            cols["zone"].append(zi)
            cols["month"].append(month + 1)
            cols["era"].append(era)
            cols["ratio"].append(json_number(d["ratio"][zi, month]))
            cols["lo"].append(json_number(d["lo"][zi, month]))
            cols["hi"].append(json_number(d["hi"][zi, month]))
            cols["widened_lo"].append(None)
            cols["widened_hi"].append(None)
            cols["n_support"].append(
                int(d["support"][zi, month])
            )

# Yearly pilot-month ratios: Era B, since this schema has no era field.
yearly_cols = {
    "zone": [], "month": [], "year": [],
    "ratio": [], "in_learning_window": []
}

for zi in range(len(zones)):
    for month in [9, 10]:  # October and November, zero-indexed
        for yi, year in enumerate(ratios.YEARS):
            numerator = V[zi, yi, month]
            denominator = M_B[zi, yi, month]

            value = (
                numerator / denominator
                if np.isfinite(numerator)
                and np.isfinite(denominator)
                and denominator > 0
                else None
            )

            yearly_cols["zone"].append(zi)
            yearly_cols["month"].append(month + 1)
            yearly_cols["year"].append(int(year))
            yearly_cols["ratio"].append(value)
            yearly_cols["in_learning_window"].append(True)

payload = {
    "schema_version": "1.0",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "mock": True,
    "unit": "VIIRS SP tile-days per MODIS SP tile-day",
    "learning_window": [2012, 2019],
    "k": float(ratios.K),
    "zones": list(zones),
    "columns": cols,
    "ratio_by_year": {"columns": yearly_cols}
}

OUT.write_text(
    json.dumps(payload, allow_nan=False, separators=(",", ":")),
    encoding="utf-8"
)

errors = validate_file("conversion", OUT)

print("Saved:", OUT)
print("Main conversion rows:", len(cols["zone"]))
print("Yearly pilot rows:", len(yearly_cols["zone"]))
print("Era A rows:", cols["era"].count("A"))
print("Era B rows:", cols["era"].count("B"))
print("Null ratios:", sum(x is None for x in cols["ratio"]))
print("Validation errors:", errors)

assert not errors, "Conversion contract validation failed."
print("PASS: conversion.json validates.")

Saved: web/public/data/conversion.json
Main conversion rows: 384
Yearly pilot rows: 256
Era A rows: 192
Era B rows: 192
Null ratios: 0
Validation errors: []
PASS: conversion.json validates.


In [88]:
from pipeline.tables import config as C

print("Allowed validation labels:", C.LABELS)
print("Presets:", C.PRESET_IDS)
print("Learning years:", ratios.YEARS)


Allowed validation labels: ['Measured', 'Estimated', 'Indicative', 'Provisional']
Presets: ['punjab', 'haryana', 'punjab_haryana']
Learning years: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]


In [89]:
import json
import numpy as np
from datetime import datetime, timezone
from pathlib import Path
from pipeline.validate import hide_a_year as validation
from pipeline.tables.contracts import validate_file

OUT = Path("web/public/data/validation.json")

# Holdout results for all three presets and all eight years.
holdout = validation.hide_a_year(
    cubes["VIIRS_SNPP"],
    cubes["MODIS_ALL"]
)

def safe_num(x):
    return float(x) if np.isfinite(x) else None

region_col, year_col = [], []
predicted_col, actual_col, range_col = [], [], []

for row in holdout:
    for month in [9, 10]:
        region_col.append(int(row["region"]))
        year_col.append(int(row["year"]))
        predicted_col.append(safe_num(row["pred"][month]))
        actual_col.append(safe_num(row["actual"][month]))
        lo = row["pred_lo"][month]
        hi = row["pred_hi"][month]
        actual = row["actual"][month]
        range_col.append(
            bool(lo <= actual <= hi)
            if np.isfinite(lo) and np.isfinite(hi)
            and np.isfinite(actual) else None
        )

# Summarize combined-region holdout errors.
summary = validation.summarize(
    holdout, months=[9, 10], preset="punjab_haryana"
)
error_table = [
    {"actual": r["actual"], "err": r["err"]}
    for r in summary
    if np.isfinite(r["actual"]) and np.isfinite(r["err"])
]
level_result = validation.error_by_level(error_table)

def check_block(owner, reviewer, status, label, result):
    return {
        "owner": owner,
        "reviewer": reviewer,
        "status": status,
        "label": label,
        "result": result
    }

pending = lambda owner, reviewer, label: check_block(
    owner, reviewer, "pending", label, None
)

payload = {
    "schema_version": "1.0",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "mock": True,

    "hide_a_year": check_block(
        "M2", "M3", "done", "Provisional",
        {"columns": {
            "region": region_col,
            "year": year_col,
            "predicted": predicted_col,
            "actual": actual_col,
            "in_range": range_col
        }}
    ),

    "error_by_level": check_block(
        "M2", "M3", "done", "Provisional",
        {
            "method": "Three activity thirds of combined-region October-November holdout years",
            "levels": {
                name: {
                    "mean_absolute_pct_error": safe_num(values[0]),
                    "mean_pct_bias": safe_num(values[1])
                }
                for name, values in level_result.items()
            }
        }
    ),

    "baselines": pending("M3", "M2", "Provisional"),
    "rank_test": pending("M3", "M2", "Provisional"),
    "seam_step": pending("M5", "M2", "Provisional"),
    "era_b_test": pending("M2", "M3", "Provisional"),
    "ratio_trend": pending("M2", "M3", "Provisional"),
    "res_sensitivity": pending("M1", "M5", "Provisional"),
    "nrt_seam": pending("M1", "M2", "Provisional"),
    "external_check": pending("M3", "M6", "Provisional"),
    "negative_control": pending("M3", "M1", "Provisional")
}

OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(
    json.dumps(payload, allow_nan=False, separators=(",", ":")),
    encoding="utf-8"
)

errors = validate_file("validation", OUT)
print("Saved:", OUT)
print("Holdout observations:", len(region_col))
print("Completed checks:", ["hide_a_year", "error_by_level"])
print("Pending checks:", 8)
print("Validation errors:", errors)

assert not errors, "Validation contract failed."
print("PASS: validation.json validates.")

Saved: web/public/data/validation.json
Holdout observations: 48
Completed checks: ['hide_a_year', 'error_by_level']
Pending checks: 8
Validation errors: []
PASS: validation.json validates.


In [90]:
from pathlib import Path
from pipeline.tables.contracts import validate_tree
import subprocess
import sys

WORK = Path("/content/emberwise_m2_work")
DATA = WORK / "web/public/data"
PROCESSED = WORK / "data/processed"

# Validate all contract files and processed tables.
report = validate_tree(DATA, PROCESSED)

print("=== CONTRACT VALIDATION ===")
failed = False
for name, errors in report.items():
    status = "PASS" if not errors else "FAIL"
    print(f"{status}: {name}")
    for error in errors:
        print("  -", error)
    failed |= bool(errors)

print("\n=== TABLE TESTS ===")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "pipeline/tables"],
    cwd=WORK,
    text=True,
    capture_output=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr[-3000:])
print("Pytest exit code:", result.returncode)

if failed or result.returncode != 0:
    print("\nQUALITY GATE: NEEDS FIXES")
else:
    print("\nQUALITY GATE: PASS")

=== CONTRACT VALIDATION ===
PASS: regions.json
PASS: coverage.json
PASS: hourly.json
PASS: drift.json
PASS: conversion.json
PASS: validation.json
PASS: relay_sim.json
PASS: tile_month_2000.json
PASS: tile_month_2001.json
PASS: tile_month_2002.json
PASS: tile_month_2003.json
PASS: tile_month_2004.json
PASS: tile_month_2005.json
PASS: tile_month_2006.json
PASS: tile_month_2007.json
PASS: tile_month_2008.json
PASS: tile_month_2009.json
PASS: tile_month_2010.json
PASS: tile_month_2011.json
PASS: tile_month_2012.json
PASS: tile_month_2013.json
PASS: tile_month_2014.json
PASS: tile_month_2015.json
PASS: tile_month_2016.json
PASS: tile_month_2017.json
PASS: tile_month_2018.json
PASS: tile_month_2019.json
PASS: tile_month_2020.json
PASS: tile_month_2021.json
PASS: tile_month_2022.json
PASS: tile_month_2023.json
PASS: tile_month_2024.json
PASS: tile_month_2025.json
PASS: tile_month_2026.json
PASS: tile_geometry.json
PASS: region_daily.json
PASS: climatology.json
PASS: periods.json
PASS: narrati

In [91]:
from pathlib import Path

ROOT = Path("/content/emberwise_m2_work")

print("=== MANIFEST REFERENCES ===")
for p in ROOT.rglob("*"):
    if not p.is_file() or any(x in p.parts for x in [".git", ".venv", "__pycache__"]):
        continue
    if p.name in {"Makefile", "pyproject.toml", "README.md", "package.json"}:
        text = p.read_text(errors="ignore")
        matches = [
            (i, line) for i, line in enumerate(text.splitlines(), 1)
            if "manifest" in line.lower() or "pytest" in line.lower() or "test" in line.lower()
        ]
        if matches:
            print(f"\n--- {p.relative_to(ROOT)} ---")
            for i, line in matches[:35]:
                print(f"{i}: {line}")

print("\n=== MANIFEST MODULES / SCRIPTS ===")
for p in ROOT.rglob("*.py"):
    if any(x in p.parts for x in [".git", ".venv", "__pycache__"]):
        continue
    text = p.read_text(errors="ignore")
    if "manifest.json" in text or "build_manifest" in text:
        print(p.relative_to(ROOT))

print("\n=== TEST FILES ===")
for p in ROOT.rglob("*"):
    if p.is_file() and ("test" in p.name.lower() or "tests" in p.parts):
        if not any(x in p.parts for x in [".git", ".venv", "__pycache__"]):
            print(p.relative_to(ROOT))

=== MANIFEST REFERENCES ===

--- .pytest_cache/README.md ---
1: # pytest cache directory #
3: This directory contains data from the pytest's cache plugin,
8: See [the docs](https://docs.pytest.org/en/stable/how-to/cache.html) for more information.

=== MANIFEST MODULES / SCRIPTS ===
tests/test_tables_stage_a.py
pipeline/tables/build.py
pipeline/tables/contracts.py

=== TEST FILES ===
conftest.py
tests/test_tables_stage_a.py


In [92]:
from pathlib import Path
import subprocess
import sys
import inspect
from pipeline.tables import build

print("=== BUILD MODULE ===")
print(inspect.getsource(build))

print("\n=== PROJECT TEST SUITE ===")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_tables_stage_a.py"],
    cwd="/content/emberwise_m2_work",
    text=True,
    capture_output=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr[-3000:])
print("Pytest exit code:", result.returncode)

=== BUILD MODULE ===
"""Build M3's tables from the hand-offs (plan s5): the same code runs on mocks and on real data.

Inputs : data/processed/tiles.parquet, data/processed/tile_day.parquet (M1),
         web/public/data/{regions,coverage,conversion}.json (M6/M1, M1, M2).
Outputs: tile_month_YYYY.json, region_daily.json, climatology.json, periods.json,
         narratives.json, tile_geometry.json, manifest.json in web/public/data.

Rules enforced here: one source of truth per day (VIIRS where it has data,
otherwise MODIS x conversion ratio); MODIS and VIIRS are never added for the
same day; a gap is null, never zero; every value carries a label.
"""
from __future__ import annotations

from pathlib import Path

import h3
import numpy as np
import polars as pl

from . import analytics as A
from . import config as C
from .contracts import CONTRACTS
from .harmonize import convert_modis, harmonize, support_factor, trust_score
from .jsonio import envelope, nan_array, qlist, read_json, sha256

In [94]:
import os
import sys
import subprocess
from pathlib import Path

ROOT = Path("/content/emberwise_m2_work")
os.chdir(ROOT)

from pipeline.tables import build
from pipeline.tables.contracts import validate_tree

# Rebuild derived outputs and manifest using the M2 conversion hand-off.
summary = build.build(
    data_dir=ROOT / "web/public/data",
    processed_dir=ROOT / "data/processed"
)
print("=== BUILD SUMMARY ===")
print(summary)

# Revalidate every contract.
print("\n=== CONTRACT VALIDATION ===")
report = validate_tree()
failed = {name: errors for name, errors in report.items() if errors}

for name, errors in report.items():
    print(("FAIL: " if errors else "PASS: ") + name)
    for error in errors:
        print("  -", error)

# Run the actual project test suite.
print("\n=== PROJECT TESTS ===")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_tables_stage_a.py"],
    cwd=ROOT,
    text=True,
    capture_output=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr[-2000:])

assert not failed, f"Contract failures: {failed}"
assert result.returncode == 0, "Project tests did not pass."
print("\nQUALITY GATE: PASS")

=== BUILD SUMMARY ===
{'as_of': '2026-11-13', 'tiles': 2777, 'zones': 16, 'months': 313, 'mock': True}

=== CONTRACT VALIDATION ===
PASS: regions.json
PASS: coverage.json
PASS: hourly.json
PASS: drift.json
PASS: conversion.json
PASS: validation.json
PASS: relay_sim.json
PASS: tile_month_2000.json
PASS: tile_month_2001.json
PASS: tile_month_2002.json
PASS: tile_month_2003.json
PASS: tile_month_2004.json
PASS: tile_month_2005.json
PASS: tile_month_2006.json
PASS: tile_month_2007.json
PASS: tile_month_2008.json
PASS: tile_month_2009.json
PASS: tile_month_2010.json
PASS: tile_month_2011.json
PASS: tile_month_2012.json
PASS: tile_month_2013.json
PASS: tile_month_2014.json
PASS: tile_month_2015.json
PASS: tile_month_2016.json
PASS: tile_month_2017.json
PASS: tile_month_2018.json
PASS: tile_month_2019.json
PASS: tile_month_2020.json
PASS: tile_month_2021.json
PASS: tile_month_2022.json
PASS: tile_month_2023.json
PASS: tile_month_2024.json
PASS: tile_month_2025.json
PASS: tile_month_2026.json


AssertionError: Project tests did not pass.

In [95]:
import json
from pathlib import Path
from pipeline.tables import config as C
from pipeline.tables.build import day_flags, day_index, month_list, ratio_tables, load_inputs
from pipeline.tables import analytics as A
import numpy as np

ROOT = Path("/content/emberwise_m2_work")
DATA = ROOT / "web/public/data"

daily = json.loads((DATA / "region_daily.json").read_text())
conv = json.loads((DATA / "conversion.json").read_text())

# 1. Inspect the 2008 MODIS-only day.
s = daily["presets"]["punjab_haryana"]
i = day_index("2008-11-03")

print("=== 2008-11-03 DAILY VALUES ===")
for key in ["harm", "lo", "hi", "modis_td", "viirs_td", "label"]:
    print(key, "=", s[key][i])

# 2. Inspect Era A conversion bounds for November.
print("\n=== ERA A CONVERSION: NOVEMBER ===")
cols = conv["columns"]
for j, (zone, month, era) in enumerate(
    zip(cols["zone"], cols["month"], cols["era"])
):
    if month == 11 and era == "A":
        print({
            "zone": conv["zones"][zone],
            "ratio": cols["ratio"][j],
            "lo": cols["lo"][j],
            "hi": cols["hi"][j],
            "widened_lo": cols["widened_lo"][j],
            "widened_hi": cols["widened_hi"][j],
            "support": cols["n_support"][j]
        })
        if j > 8:
            break

# 3. Inspect early tile-month data and trust.
print("\n=== EARLY TILE-MONTH OUTPUTS ===")
for year in [2001, 2005]:
    obj = json.loads((DATA / f"tile_month_{year}.json").read_text())
    trust = obj["columns"]["trust"]
    print(
        year,
        "rows:", len(obj["columns"]["tile"]),
        "non-null trust:", sum(x is not None for x in trust),
        "non-null raw MODIS:", sum(x is not None for x in obj["columns"]["raw_modis"]),
        "non-null harmonized:", sum(x is not None for x in obj["columns"]["harm"])
    )

# 4. Inspect the source data and coverage for those years.
inp = load_inputs(DATA, ROOT / "data/processed")
n_days = day_index(inp["as_of"]) + 1
flags = day_flags(inp["coverage"], n_days)

print("\n=== COVERAGE FLAGS ===")
for date in ["2001-11-03", "2005-11-03", "2008-11-03"]:
    d = day_index(date)
    print(date, {
        "MODIS": bool(flags["modis"][d]),
        "VIIRS": bool(flags["vday"][d]),
        "MODIS_only": bool(flags["mday"][d])
    })

print("\n=== TRUST FUNCTION ===")
import inspect
from pipeline.tables import harmonize as H
print(inspect.getsource(H.trust_score))

=== 2008-11-03 DAILY VALUES ===
harm = 1478.4
lo = None
hi = None
modis_td = 848
viirs_td = None
label = 1

=== ERA A CONVERSION: NOVEMBER ===
{'zone': '833d10fffffffff', 'ratio': 3.1465306966691275, 'lo': 2.9954177649587184, 'hi': 3.386134005151679, 'widened_lo': None, 'widened_hi': None, 'support': 1040}

=== EARLY TILE-MONTH OUTPUTS ===
2001 rows: 6777 non-null trust: 0 non-null raw MODIS: 6777 non-null harmonized: 6777
2005 rows: 7230 non-null trust: 0 non-null raw MODIS: 7230 non-null harmonized: 7230

=== COVERAGE FLAGS ===
2001-11-03 {'MODIS': True, 'VIIRS': False, 'MODIS_only': True}
2005-11-03 {'MODIS': True, 'VIIRS': False, 'MODIS_only': True}
2008-11-03 {'MODIS': True, 'VIIRS': False, 'MODIS_only': True}

=== TRUST FUNCTION ===
def trust_score(harm, lo, hi, f_support, f_era):
    """Trust 0-100 = 100 x (1 - range width / value) x f_support x f_era, floored at 0.

    Returns NaN (shown as n/a, no activity) where the value is 0 or missing.
    """
    harm = np.asarray(harm, 

In [96]:
import json
import subprocess
import sys
from pathlib import Path
from pipeline.tables import build
from pipeline.tables.contracts import validate_tree

ROOT = Path("/content/emberwise_m2_work")
DATA = ROOT / "web/public/data"
path = DATA / "conversion.json"

# Preserve the current file before making the explicit provisional change.
backup = DATA / "conversion_before_bounds.json"
backup.write_text(path.read_text())

conv = json.loads(path.read_text())
cols = conv["columns"]

filled = 0
for i, (lo, hi) in enumerate(zip(cols["lo"], cols["hi"])):
    if (cols["widened_lo"][i] is None
            and cols["widened_hi"][i] is None
            and lo is not None and hi is not None):
        # Use fitted percentile bounds; apply no extra empirical widening.
        cols["widened_lo"][i] = lo
        cols["widened_hi"][i] = hi
        filled += 1

path.write_text(json.dumps(conv, allow_nan=False, separators=(",", ":")))
print("Bounds populated from fitted percentile ranges:", filled)

# Rebuild dependent outputs and manifest.
summary = build.build(DATA, ROOT / "data/processed")
print("Build:", summary)

# Validate all output contracts.
report = validate_tree()
failures = {k: v for k, v in report.items() if v}
print("Contract failures:", failures)

# Run the actual project test suite.
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_tables_stage_a.py"],
    cwd=ROOT, text=True, capture_output=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr[-2000:])
print("Pytest exit code:", result.returncode)

assert not failures, failures
assert result.returncode == 0, "Tests still need investigation."
print("QUALITY GATE: PASS")

Bounds populated from fitted percentile ranges: 384


StopIteration: 

In [97]:
from pathlib import Path
from pipeline.tables import build
from pipeline.tables.contracts import CONTRACTS

ROOT = Path("/content/emberwise_m2_work")
DATA = ROOT / "web/public/data"

unmatched = []

for p in sorted(DATA.rglob("*")):
    if not p.is_file() or p.name == "manifest.json":
        continue

    rel = p.relative_to(DATA).as_posix()

    if rel.startswith("hotspots/"):
        continue

    matches = [
        name for name, (_, pattern) in CONTRACTS.items()
        if Path(rel).match(pattern)
    ]

    if not matches:
        unmatched.append(rel)

print("Unrecognized files:", len(unmatched))
for name in unmatched:
    print(" -", name)

print("\nContract patterns:")
for name, (_, pattern) in CONTRACTS.items():
    print(name, "->", pattern)

Unrecognized files: 1
 - conversion_before_bounds.json

Contract patterns:
regions -> regions.json
coverage -> coverage.json
hourly -> hourly.json
drift -> drift.json
conversion -> conversion.json
validation -> validation.json
relay_sim -> relay_sim.json
tile_month -> tile_month_*.json
tile_geometry -> tile_geometry.json
region_daily -> region_daily.json
climatology -> climatology.json
periods -> periods.json
narratives -> narratives.json
manifest -> manifest.json


In [98]:
from pathlib import Path
import shutil
import os

ROOT = Path("/content/emberwise_m2_work")
DATA = ROOT / "web/public/data"
BACKUP = ROOT / "backups/conversion_before_bounds.json"

os.chdir(ROOT)

source = DATA / "conversion_before_bounds.json"
BACKUP.parent.mkdir(parents=True, exist_ok=True)

if source.exists():
    shutil.move(str(source), str(BACKUP))

print("Backup preserved:", BACKUP.exists())

from pipeline.tables import build

summary = build.build(DATA, ROOT / "data/processed")
print("Build successful:", summary)

Backup preserved: True
Build successful: {'as_of': '2026-11-13', 'tiles': 2777, 'zones': 16, 'months': 313, 'mock': True}


In [99]:
import subprocess
import sys
from pathlib import Path
from pipeline.tables.contracts import validate_tree

ROOT = Path("/content/emberwise_m2_work")

print("=== CONTRACT CHECK ===")
report = validate_tree()
failed = {k: v for k, v in report.items() if v}

print("Files checked:", len(report))
print("Contract failures:", failed)

print("\n=== PROJECT TESTS ===")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_tables_stage_a.py"],
    cwd=ROOT,
    text=True,
    capture_output=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr[-2000:])

print("Exit code:", result.returncode)

assert not failed, "Some contracts failed."
assert result.returncode == 0, "Some project tests failed."

print("\nM2 MOCK-DATA QUALITY GATE: PASS")


=== CONTRACT CHECK ===
Files checked: 45
Contract failures: {}

=== PROJECT TESTS ===
.........................................                                [100%]
41 passed in 55.55s

Exit code: 0

M2 MOCK-DATA QUALITY GATE: PASS


In [100]:
%%writefile /content/emberwise_m2_work/pipeline/calibrate/ratios.py
"""M2: MODIS -> VIIRS conversion ratios. Cubes are [region, zone, year, month]; an uncovered month is NaN, never 0."""
import numpy as np

K = 20
YEARS = list(range(2012, 2020))
BASE_REGIONS = ["punjab", "haryana"]
PRESETS = {"punjab": [0], "haryana": [1], "punjab_haryana": [0, 1]}   # same order as the contract


def month_ok(coverage, sensors, years):
    """{sensor: bool [year, month]}: True only if SP coverage spans the whole calendar month (adjacent intervals are merged)."""
    from datetime import date, timedelta
    epoch = date.fromisoformat(coverage["epoch"])
    c = coverage["columns"]
    sp = coverage["products"].index("SP")
    out = {}
    for s in sensors:
        si = coverage["sensor_sets"].index(s)
        iv = sorted((c["start"][i], c["end"][i]) for i in range(len(c["sensor_set"]))
                    if c["sensor_set"][i] == si and c["product"][i] == sp and c["has_data"][i])
        merged = []
        for a, b in iv:
            if merged and a <= merged[-1][1] + 1:
                merged[-1][1] = max(merged[-1][1], b)
            else:
                merged.append([a, b])
        ok = np.zeros((len(years), 12), bool)
        for yi, y in enumerate(years):
            for m in range(12):
                first = (date(y, m + 1, 1) - epoch).days
                last = ((date(y + 1, 1, 1) if m == 11 else date(y, m + 2, 1)) - epoch).days - 1
                ok[yi, m] = any(a <= first and b >= last for a, b in merged)
        out[s] = ok
    return out


def joint_ok(ok):
    """A month counts only if EVERY sensor covered it fully."""
    return np.logical_and.reduce(list(ok.values()))


def build_cubes(path, coverage_path="web/public/data/coverage.json",
                sensors=("VIIRS_SNPP", "MODIS_ALL", "TERRA"), years=YEARS):
    """tile_day.parquet -> {sensor: [region, zone, year, month]} burning tile-days (SP only); joint-uncovered months NaN."""
    import json
    import polars as pl
    years = list(years)
    joint = joint_ok(month_ok(json.load(open(coverage_path)), sensors, years))
    td = pl.read_parquet(path).with_columns(pl.col("region", "sensor_set", "product").cast(pl.String))
    zones = sorted(td.filter(pl.col("region").is_in(BASE_REGIONS))["zone_r3"].unique().to_list())
    zi = {z: i for i, z in enumerate(zones)}
    cubes = {s: np.zeros((len(BASE_REGIONS), len(zones), len(years), 12)) for s in sensors}
    g = (td.filter(pl.col("region").is_in(BASE_REGIONS) & (pl.col("product") == "SP")
                   & pl.col("sensor_set").is_in(list(sensors))
                   & pl.col("date").dt.year().is_between(years[0], years[-1]))
           .group_by("region", "zone_r3", "sensor_set", pl.col("date").dt.year().alias("y"),
                     pl.col("date").dt.month().alias("m"))
           .agg(pl.col("n_tile_day").sum().alias("n")))
    for r, z, s, y, m, n in g.iter_rows():
        cubes[s][BASE_REGIONS.index(r), zi[z], y - years[0], m - 1] = n
    for cube in cubes.values():
        cube[:, :, ~joint] = np.nan
    print("months dropped (any sensor uncovered):", [(years[a], int(b) + 1) for a, b in zip(*np.where(~joint))])
    return cubes, zones


def pool(cube, parts=None):
    """Sum regions -> [zone, year, month]; NaN only where every region is NaN."""
    c = cube if parts is None else cube[parts]
    out = np.nansum(c, axis=0)
    out[np.all(np.isnan(c), axis=0)] = np.nan
    return out


def fit_ratio(V, M, k=K):
    """V, M: [zone, year, month]. Shrunk ratio [zone, month], yearly [zone, year, month], region [month]. NaN = no data."""
    V, M = np.asarray(V, float), np.asarray(M, float)
    valid = np.isfinite(V) & np.isfinite(M)
    Vv, Mv = np.where(valid, V, 0.0), np.where(valid, M, 0.0)
    v_reg, m_reg = Vv.sum(axis=(0, 1)), Mv.sum(axis=(0, 1))
    r_region = np.divide(v_reg, m_reg, out=np.full(V.shape[2], np.nan), where=(m_reg > 0) & (valid.sum(axis=(0, 1)) > 0))
    prior = np.nan_to_num(r_region, nan=0.0)
    has = np.isfinite(r_region)[None, :]
    r_hat = np.where(valid.any(axis=1) & has, (Vv.sum(axis=1) + k * prior) / (Mv.sum(axis=1) + k), np.nan)
    yearly = np.where(valid & has[:, None, :], (Vv + k * prior) / (Mv + k), np.nan)
    return r_hat, yearly, r_region


def ratio_range(r_hat, yearly):
    """Base range: 10th-90th percentile of yearly ratios, always containing r_hat."""
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        lo, hi = np.nanpercentile(yearly, 10, axis=1), np.nanpercentile(yearly, 90, axis=1)
    return np.minimum(lo, r_hat), np.maximum(hi, r_hat)

Overwriting /content/emberwise_m2_work/pipeline/calibrate/ratios.py


In [101]:
%%writefile /content/emberwise_m2_work/pipeline/validate/hide_a_year.py
"""M2: hide-a-year. Learn on the other years (region ratio recomputed without the hidden year), predict from MODIS alone."""
import numpy as np
from pipeline.calibrate.ratios import fit_ratio, ratio_range, pool, PRESETS, YEARS

OCT_NOV = [9, 10]


def hide_a_year(V, M, years=YEARS, k=20):
    V, M = np.asarray(V, float), np.asarray(M, float)
    if V.shape != M.shape or V.ndim != 4 or V.shape[2] != len(years):
        raise ValueError("V and M must be [region, zone, year, month] with one year per entry in `years`.")
    valid = np.isfinite(V) & np.isfinite(M)
    Vc, Mc = np.where(valid, V, 0.0), np.where(valid, M, 0.0)
    Vz, Mz = pool(np.where(valid, V, np.nan)), pool(np.where(valid, M, np.nan))
    rows = []
    for i, y in enumerate(years):
        keep = [j for j in range(len(years)) if j != i]
        r_hat, yearly, _ = fit_ratio(Vz[:, keep], Mz[:, keep], k)
        lo, hi = ratio_range(r_hat, yearly)
        tv, tm = np.nansum(Vz[:, keep]), np.nansum(Mz[:, keep])
        g = tv / tm if tm > 0 else np.nan
        for pi, (pid, parts) in enumerate(PRESETS.items()):
            m = Mc[parts].sum(axis=0)[:, i]
            rows.append(dict(region=pi, preset=pid, year=y, actual=Vc[parts].sum(axis=0)[:, i].sum(axis=0),
                             pred=np.nansum(m * r_hat, axis=0), pred_lo=np.nansum(m * lo, axis=0),
                             pred_hi=np.nansum(m * hi, axis=0), base_none=m.sum(axis=0), base_global=m.sum(axis=0) * g))
    return rows


def pct_err(p, a):
    return 100.0 * (p - a) / a if a else np.nan


def summarize(rows, months=None, preset="punjab_haryana"):
    months = OCT_NOV if months is None else months
    out = []
    for r in rows:
        if r["preset"] != preset:
            continue
        a, p = r["actual"][months].sum(), r["pred"][months].sum()
        out.append(dict(year=r["year"], actual=a, pred=p, err=pct_err(p, a),
                        in_range=bool(r["pred_lo"][months].sum() <= a <= r["pred_hi"][months].sum()),
                        err_none=pct_err(r["base_none"][months].sum(), a),
                        err_global=pct_err(r["base_global"][months].sum(), a)))
    return out


def error_by_level(table):
    t = sorted(table, key=lambda r: r["actual"]); n = len(t)
    c1, c2 = round(n / 3), round(2 * n / 3)
    return {k: (float(np.nanmean([abs(r["err"]) for r in g])), float(np.nanmean([r["err"] for r in g])))
            for k, g in (("low", t[:c1]), ("mid", t[c1:c2]), ("high", t[c2:])) if g}

Overwriting /content/emberwise_m2_work/pipeline/validate/hide_a_year.py


In [102]:
%%writefile /content/emberwise_m2_work/pipeline/validate/hide_a_year.py
"""M2: hide-a-year. Learn on the other years (region ratio recomputed without the hidden year), predict from MODIS alone."""
import numpy as np
from pipeline.calibrate.ratios import fit_ratio, ratio_range, pool, PRESETS, YEARS

OCT_NOV = [9, 10]


def hide_a_year(V, M, years=YEARS, k=20):
    V, M = np.asarray(V, float), np.asarray(M, float)
    if V.shape != M.shape or V.ndim != 4 or V.shape[2] != len(years):
        raise ValueError("V and M must be [region, zone, year, month] with one year per entry in `years`.")
    valid = np.isfinite(V) & np.isfinite(M)
    Vc, Mc = np.where(valid, V, 0.0), np.where(valid, M, 0.0)
    Vz, Mz = pool(np.where(valid, V, np.nan)), pool(np.where(valid, M, np.nan))
    rows = []
    for i, y in enumerate(years):
        keep = [j for j in range(len(years)) if j != i]
        r_hat, yearly, _ = fit_ratio(Vz[:, keep], Mz[:, keep], k)
        lo, hi = ratio_range(r_hat, yearly)
        tv, tm = np.nansum(Vz[:, keep]), np.nansum(Mz[:, keep])
        g = tv / tm if tm > 0 else np.nan
        for pi, (pid, parts) in enumerate(PRESETS.items()):
            m = Mc[parts].sum(axis=0)[:, i]
            rows.append(dict(region=pi, preset=pid, year=y, actual=Vc[parts].sum(axis=0)[:, i].sum(axis=0),
                             pred=np.nansum(m * r_hat, axis=0), pred_lo=np.nansum(m * lo, axis=0),
                             pred_hi=np.nansum(m * hi, axis=0), base_none=m.sum(axis=0), base_global=m.sum(axis=0) * g))
    return rows


def pct_err(p, a):
    return 100.0 * (p - a) / a if a else np.nan


def summarize(rows, months=None, preset="punjab_haryana"):
    months = OCT_NOV if months is None else months
    out = []
    for r in rows:
        if r["preset"] != preset:
            continue
        a, p = r["actual"][months].sum(), r["pred"][months].sum()
        out.append(dict(year=r["year"], actual=a, pred=p, err=pct_err(p, a),
                        in_range=bool(r["pred_lo"][months].sum() <= a <= r["pred_hi"][months].sum()),
                        err_none=pct_err(r["base_none"][months].sum(), a),
                        err_global=pct_err(r["base_global"][months].sum(), a)))
    return out


def error_by_level(table):
    t = sorted(table, key=lambda r: r["actual"]); n = len(t)
    c1, c2 = round(n / 3), round(2 * n / 3)
    return {k: (float(np.nanmean([abs(r["err"]) for r in g])), float(np.nanmean([r["err"] for r in g])))
            for k, g in (("low", t[:c1]), ("mid", t[c1:c2]), ("high", t[c2:])) if g}

Overwriting /content/emberwise_m2_work/pipeline/validate/hide_a_year.py


In [103]:
%%writefile /content/emberwise_m2_work/pipeline/validate/checks.py
"""M2: tested widening, hide-a-year block, error by level, era-B test, ratio trend. Arrays: [region, zone, year, month]."""
import numpy as np
from scipy import stats
from pipeline.calibrate.ratios import fit_ratio, pool, PRESETS, YEARS

PILOT = [9, 10]            # Oct, Nov as month indexes
MIN_ACTUAL = 10            # ignore zone-months with fewer than this many VIIRS tile-days (noise-dominated)
TREND_P, TREND_REL = 0.10, 2.0


def p80(errs):
    e = np.abs(np.asarray(errs, float))
    return float(np.percentile(e, 80)) if len(e) else None


def cell_errors(V, M, years=YEARS, k=20, months=PILOT, min_actual=MIN_ACTUAL):
    """Zone-month relative errors (pred-actual)/actual for the hidden year, Oct-Nov, learned without that year."""
    Vz, Mz = pool(V), pool(M)
    out = []
    for i in range(len(years)):
        keep = [j for j in range(len(years)) if j != i]
        r, _, _ = fit_ratio(Vz[:, keep], Mz[:, keep], k)
        for m in months:
            a, p = Vz[:, i, m], Mz[:, i, m] * r[:, m]
            ok = (a >= min_actual) & np.isfinite(p)
            out.extend(((p[ok] - a[ok]) / a[ok]).tolist())
    return np.array(out)


def block(result, mock, label="Measured", reviewer="M3"):
    return {"owner": "M2", "reviewer": reviewer, "status": "pending" if mock else "done",
            "label": label, "result": result}


def hide_a_year_block(rows, w, mock):
    """Contract columns: region (preset index), year, predicted, actual (all-month totals), in_range (widened by w)."""
    cols = {k: [] for k in ("region", "year", "predicted", "actual", "in_range")}
    for r in rows:
        pred, act = float(np.nansum(r["pred"])), float(np.nansum(r["actual"]))
        lo, hi = float(np.nansum(r["pred_lo"])) * (1 - w), float(np.nansum(r["pred_hi"])) * (1 + w)
        cols["region"].append(int(r["region"])); cols["year"].append(int(r["year"]))
        cols["predicted"].append(round(pred, 1)); cols["actual"].append(round(act, 1))
        cols["in_range"].append(bool(lo <= act <= hi))
    return block({"columns": cols}, mock)


def error_by_level_block(table, w, mock, preset="punjab_haryana"):
    t = sorted(table, key=lambda r: r["actual"]); n = len(t)
    c1, c2 = round(n / 3), round(2 * n / 3)
    levels = {}
    for name, g in (("low", t[:c1]), ("mid", t[c1:c2]), ("high", t[c2:])):
        if g:
            e = np.array([r["err"] for r in g], float)
            levels[name] = {"years": [int(r["year"]) for r in g],
                            "mean_abs_error_pct": round(float(np.nanmean(np.abs(e))), 2),
                            "bias_pct": round(float(np.nanmean(e)), 2)}
    return block({"preset": preset, "months": [10, 11],
                  "metric": "percent error of Oct-Nov tile-days, hidden year, grouped by actual activity (3/2/3 years)",
                  "levels": levels, "widening_used": round(w, 4),
                  "note": "Eight years only; tail error is reported as found."}, mock)


def era_b_test(T, Mt, rA, rB, years, mock, months=PILOT, min_actual=MIN_ACTUAL):
    """2003-2011: does Terra x r_A agree with (Terra U Aqua) x r_B? Consistency check; no VIIRS exists in these years."""
    T, Mt = pool(T), pool(Mt)
    rel, by_year = [], []
    for yi, y in enumerate(years):
        sa = sb = 0.0
        for m in months:
            a, b = T[:, yi, m] * rA[:, m], Mt[:, yi, m] * rB[:, m]
            ok = (b >= min_actual) & np.isfinite(a) & np.isfinite(b)
            rel.extend(((a[ok] - b[ok]) / b[ok]).tolist()); sa += a[ok].sum(); sb += b[ok].sum()
        by_year.append({"year": int(y), "signed_pct": round(100 * (sa - sb) / sb, 2) if sb > 0 else None})
    rel = np.array(rel)
    e_b = p80(rel)
    res = {"years": [int(years[0]), int(years[-1])], "months": [10, 11],
           "metric": "|Terra x r_A - MODIS_ALL x r_B| / (MODIS_ALL x r_B), zone-month",
           "n_cells": int(len(rel)),
           "p80_abs_rel_diff": None if e_b is None else round(e_b, 4),
           "median_signed_rel_diff": None if not len(rel) else round(float(np.median(rel)), 4),
           "by_year": by_year,
           "note": "Consistency check between the two conversions; no VIIRS exists in these years, so it is not a test against truth."}
    return block(res, mock, label="Estimated"), e_b


def ratio_trend_block(V, M, years, mock, months=PILOT):
    ys = np.array(years)
    def one(vv, mm):
        y = np.where(mm > 0, vv / np.where(mm > 0, mm, 1), np.nan); ok = np.isfinite(y)
        if ok.sum() < 5:
            return None
        slope = stats.theilslopes(y[ok], ys[ok])[0]
        tau, p = stats.kendalltau(ys[ok], y[ok])
        rel = 100 * slope / np.median(y[ok])
        return dict(n_years=int(ok.sum()), median_ratio=round(float(np.median(y[ok])), 3),
                    slope_per_year=round(float(slope), 4), rel_slope_pct_per_year=round(float(rel), 2),
                    kendall_tau=round(float(tau), 3), p_value=round(float(p), 3),
                    clear_trend=bool(p < TREND_P and abs(rel) >= TREND_REL))
    rows = []
    for pid, parts in PRESETS.items():
        Vp, Mp = np.nansum(pool(V, parts), axis=0), np.nansum(pool(M, parts), axis=0)
        for m in months:
            r = one(Vp[:, m], Mp[:, m])
            rows.append({"preset": pid, "month": m + 1, **(r or {"n_years": 0, "clear_trend": False})})
    Vz, Mz = pool(V), pool(M)
    tested = flagged = 0
    for z in range(Vz.shape[0]):
        for m in months:
            r = one(Vz[z, :, m], Mz[z, :, m])
            if r:
                tested += 1; flagged += r["clear_trend"]
    res = {"method": "Theil-Sen slope and Kendall tau on yearly VIIRS/MODIS tile-day ratio, learning window, Oct and Nov",
           "rule": f"clear trend if Kendall p < {TREND_P} and |relative slope| >= {TREND_REL}% per year",
           "years": [int(y) for y in years], "rows": rows,
           "zones_tested": tested, "zones_with_clear_trend": int(flagged),
           "clear_trend": bool(any(r["clear_trend"] for r in rows)),
           "note": "Eight years give low power; zone-level flags are a screen, not proof."}
    return block(res, mock)

Overwriting /content/emberwise_m2_work/pipeline/validate/checks.py


In [104]:
%%writefile /content/emberwise_m2_work/pipeline/calibrate/conversion.py
"""M2: write conversion.json in the frozen v1.0 shape with tested widening."""
import json
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
from pipeline.calibrate.ratios import fit_ratio, ratio_range, pool, K

WINDOW = (2012, 2019)
PILOT_MONTHS = [10, 11]
Q = 3


def _r(v):
    return None if not np.isfinite(v) else round(float(v), Q)


def build_conversion(cubes, zones, years, widen, mock, k=K):
    """cubes over `years` (2012..2026). widen = {"A": wA, "B": wB} from tested error (no defaults on purpose)."""
    nL = WINDOW[1] - WINDOW[0] + 1
    V = pool(cubes["VIIRS_SNPP"])
    cols = {c: [] for c in ("zone", "month", "era", "ratio", "lo", "hi", "widened_lo", "widened_hi", "n_support")}
    for era, mset in (("A", "TERRA"), ("B", "MODIS_ALL")):
        M = pool(cubes[mset])
        r_hat, yearly, _ = fit_ratio(V[:, :nL], M[:, :nL], k)
        lo, hi = ratio_range(r_hat, yearly)
        w = widen[era]
        for z in range(len(zones)):
            for m in range(12):
                cols["zone"].append(z); cols["month"].append(m + 1); cols["era"].append(era)
                for key, v in (("ratio", r_hat[z, m]), ("lo", lo[z, m]), ("hi", hi[z, m]),
                               ("widened_lo", lo[z, m] * (1 - w)), ("widened_hi", hi[z, m] * (1 + w))):
                    cols[key].append(_r(v))
                cols["n_support"].append(int(np.nansum(M[z, :nL, m])))
    Mb = pool(cubes["MODIS_ALL"])
    rby = {c: [] for c in ("zone", "month", "year", "ratio", "in_learning_window")}
    for z in range(len(zones)):
        for m in PILOT_MONTHS:
            for yi, y in enumerate(years):
                a, b = V[z, yi, m - 1], Mb[z, yi, m - 1]
                r = a / b if np.isfinite(a) and np.isfinite(b) and b > 0 else np.nan
                rby["zone"].append(z); rby["month"].append(m); rby["year"].append(int(y))
                rby["ratio"].append(_r(r)); rby["in_learning_window"].append(bool(y <= WINDOW[1]))
    return {"schema_version": "1.0", "generated_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "mock": mock, "unit": "VIIRS tile-days per MODIS tile-day", "learning_window": list(WINDOW),
            "k": k, "zones": zones, "columns": cols, "ratio_by_year": {"columns": rby}}


def save(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, allow_nan=False, separators=(",", ":")), encoding="utf-8")
    return path

Writing /content/emberwise_m2_work/pipeline/calibrate/conversion.py


In [105]:
import os, sys, numpy as np
os.chdir("/content/emberwise_m2_work")
for n in list(sys.modules):
    if n == "pipeline" or n.startswith("pipeline."):
        del sys.modules[n]
sys.path.insert(0, "/content/emberwise_m2_work")
import pipeline.calibrate.ratios as R, pipeline.validate.hide_a_year as H
print("loaded from:", R.__file__)

cubes, zones = R.build_cubes("data/processed/tile_day.parquet")
tot = {s: int(np.nansum(c)) for s, c in cubes.items()}
print(tot)
assert tot == {"VIIRS_SNPP": 341629, "MODIS_ALL": 196070, "TERRA": 108083}

r, _, _ = R.fit_ratio(R.pool(cubes["VIIRS_SNPP"]), R.pool(cubes["MODIS_ALL"]))
pct = list(np.nanpercentile(r, [10, 50, 90]).round(3)); print(pct)
assert pct == [1.645, 1.747, 1.888]

rows = H.hide_a_year(cubes["VIIRS_SNPP"], cubes["MODIS_ALL"])
tab = H.summarize(rows)
print(len(rows), "rows | 2012 predicted:", round(tab[0]["pred"], 4))
assert abs(tab[0]["pred"] - 27016.26399667042) < 1e-3
print("PASS: saved code reproduces your earlier numbers")

loaded from: /content/emberwise_m2_work/pipeline/calibrate/ratios.py
months dropped (any sensor uncovered): [(2012, 1)]
{'VIIRS_SNPP': 341629, 'MODIS_ALL': 196070, 'TERRA': 108083}
[np.float64(1.645), np.float64(1.747), np.float64(1.888)]
24 rows | 2012 predicted: 27016.264
PASS: saved code reproduces your earlier numbers


In [106]:
import json
from pathlib import Path
import pipeline.calibrate.conversion as CV
import pipeline.validate.checks as K
from pipeline.tables.contracts import validate_file

MOCK = True          # set False ONLY when M1's real tile_day.parquet and coverage.json are in place
SRC = "data/processed/tile_day.parquet"
DATA = Path("web/public/data")

V4, M4, T4 = cubes["VIIRS_SNPP"], cubes["MODIS_ALL"], cubes["TERRA"]
rows = H.hide_a_year(V4, M4); tab = H.summarize(rows)

errs = K.cell_errors(V4, M4); wB = K.p80(errs)
cubesB, _ = R.build_cubes(SRC, sensors=("MODIS_ALL", "TERRA"), years=list(range(2003, 2012)))
rA = R.fit_ratio(R.pool(V4), R.pool(T4))[0]
rB = R.fit_ratio(R.pool(V4), R.pool(M4))[0]
eb_block, eB = K.era_b_test(cubesB["TERRA"], cubesB["MODIS_ALL"], rA, rB, list(range(2003, 2012)), MOCK)
if wB is None or eB is None:
    raise ValueError("No usable cells for the widening; check the cubes first.")
widen = {"B": wB, "A": wB + eB}
trend = K.ratio_trend_block(V4, M4, R.YEARS, MOCK)

years_all = list(range(2012, 2027))
cubes_all, zones = R.build_cubes(SRC, years=years_all)
conv = CV.build_conversion(cubes_all, zones, years_all, widen, MOCK)
CV.save(conv, DATA / "conversion.json")

orig = Path("/content/emberwise/web/public/data/validation.json")      # M3's original: keeps M3/M5 blocks and owners
val = json.load(open(orig if orig.exists() else DATA / "validation.json"))
val["hide_a_year"] = K.hide_a_year_block(rows, wB, MOCK)
val["error_by_level"] = K.error_by_level_block(tab, wB, MOCK)
val["era_b_test"] = eb_block
val["ratio_trend"] = trend
(DATA / "validation.json").write_text(json.dumps(val, allow_nan=False, separators=(",", ":")), encoding="utf-8")

for name in ("conversion", "validation"):
    e = validate_file(name, DATA / f"{name}.json"); print(name, "PASS" if not e else e); assert not e

c, rb, hc = conv["columns"], conv["ratio_by_year"]["columns"], val["hide_a_year"]["result"]["columns"]
assert all(a < b and d > e for a, b, d, e in zip(c["widened_lo"], c["lo"], c["widened_hi"], c["hi"]) if None not in (a, b, d, e))
print("widening  B:", round(wB, 4), "| A:", round(widen["A"], 4), "(e_B =", round(eB, 4), ")  -> widened range is strictly wider")
print("conversion rows:", len(c["zone"]), "| ratio_by_year rows:", len(rb["zone"]), "| years:", min(rb["year"]), "-", max(rb["year"]))
print("hide-a-year rows:", len(hc["year"]), "| unique (region, year):", len(set(zip(hc["region"], hc["year"]))))
print("error by level:", {k: (v["mean_abs_error_pct"], v["bias_pct"]) for k, v in val["error_by_level"]["result"]["levels"].items()})
print("ratio trend clear:", trend["result"]["clear_trend"])

months dropped (any sensor uncovered): [(2005, 7)]
months dropped (any sensor uncovered): [(2012, 1), (2026, 9), (2026, 10), (2026, 11), (2026, 12)]
conversion PASS
validation PASS
widening  B: 0.0727 | A: 0.154 (e_B = 0.0812 )  -> widened range is strictly wider
conversion rows: 384 | ratio_by_year rows: 480 | years: 2012 - 2026
hide-a-year rows: 24 | unique (region, year): 24
error by level: {'low': (0.57, -0.2), 'mid': (0.72, -0.02), 'high': (0.2, 0.2)}
ratio trend clear: False


In [109]:
import subprocess
from pipeline.tables import build
from pipeline.tables.contracts import validate_tree
ROOT = Path("/content/emberwise_m2_work")
print("Build:", build.build(ROOT / "web/public/data", ROOT / "data/processed"))
report = validate_tree(ROOT / "web/public/data", ROOT / "data/processed")
failed = {k: v for k, v in report.items() if v}
print("Contract failures:", failed)
res = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_tables_stage_a.py"],
                     cwd=ROOT, text=True, capture_output=True)
print(res.stdout[-1500:])
assert not failed and res.returncode == 0, "Gate failed: paste me the output above."
print("M2 GATE: PASS (M3's own tests, with real tested widening)")

Build: {'as_of': '2026-11-13', 'tiles': 2777, 'zones': 16, 'months': 313, 'mock': True}
Contract failures: {}
.........................................                                [100%]
41 passed in 62.13s (0:01:02)

M2 GATE: PASS (M3's own tests, with real tested widening)


In [108]:
import shutil
from google.colab import files
out = Path("/content/send_to_M3"); shutil.rmtree(out, ignore_errors=True); out.mkdir()
shutil.copy(DATA / "conversion.json", out / "conversion.json")
val = json.load(open(DATA / "validation.json"))
frag = {k: val[k] for k in ("hide_a_year", "error_by_level", "era_b_test", "ratio_trend")}
(out / "m2_validation_blocks.json").write_text(json.dumps(frag, allow_nan=False, separators=(",", ":")))

msg = f"""*From Srishti (M2)*
Status: built on MOCK data (mock: true). The structure is final; numbers are replaced after M1's hand-off at H8.

*Ready now*
- conversion.json: era, ratio, lo, hi, widened_lo, widened_hi, n_support (384 rows), plus ratio_by_year for Oct and Nov, 2012-2026 (480 rows; in_learning_window marks 2012-2019). Widening comes from tested error: era B {wB:.1%} (80th percentile of hide-a-year zone-month error), era A {widen['A']:.1%} (adds the era-B test error {eB:.1%}).
- m2_validation_blocks.json: merge only these four keys: hide_a_year, error_by_level, era_b_test, ratio_trend. hide_a_year has 24 rows (3 presets x 8 years, one row per region-year). predicted and actual are totals over the hidden year, and in_range uses the widened range. Use it for the baseline comparison.

*Stage D (about H16), not ready yet*
- relay_sim: needs M1's NOAA-20 / NOAA-21 SP and NRT files. The relay_sim.json in the repo is still your mock. I will replace it at H16. Until then keep relay months Provisional in Local Normal and the sentences.
- nrt_seam: same dependency.
- Still waiting on your answer for schema 1.1: nrt_factor, relay_align and relay_test.
"""
(out / "message_to_M3.txt").write_text(msg); print(msg)
shutil.make_archive("/content/send_to_M3", "zip", out)
shutil.copytree("/content/emberwise_m2_work", "/content/drive/MyDrive/emberwise_m2_work_backup", dirs_exist_ok=True)
files.download("/content/send_to_M3.zip")

*From Srishti (M2)*
Status: built on MOCK data (mock: true). The structure is final; numbers are replaced after M1's hand-off at H8.

*Ready now*
- conversion.json: era, ratio, lo, hi, widened_lo, widened_hi, n_support (384 rows), plus ratio_by_year for Oct and Nov, 2012-2026 (480 rows; in_learning_window marks 2012-2019). Widening comes from tested error: era B 7.3% (80th percentile of hide-a-year zone-month error), era A 15.4% (adds the era-B test error 8.1%).
- m2_validation_blocks.json: merge only these four keys: hide_a_year, error_by_level, era_b_test, ratio_trend. hide_a_year has 24 rows (3 presets x 8 years, one row per region-year). predicted and actual are totals over the hidden year, and in_range uses the widened range. Use it for the baseline comparison.

*Stage D (about H16), not ready yet*
- relay_sim: needs M1's NOAA-20 / NOAA-21 SP and NRT files. The relay_sim.json in the repo is still your mock. I will replace it at H16. Until then keep relay months Provisional in Loca

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>